<a id="Home"></a>
# VCF Depot Quick Setup and Maintenance (Last Update: 2026-05-27)
### Version 2026-May-27

This notebook is used to help document and automate a variety of tasks that you might need to perform on this VCF depot.
They are not intended to be run sequentially.

Tasks included in this notebook are organized in the following groups:
Tasks included in this notebook include:
# Before you Begin
- [Why we use Jupyter Notebooks](#Why_JL)
- [How to use the Jupyter Notebook](#Using_JL)
- [Providing Feedback and Sharing](#Sharing_JL)
- [Task Flowchart](#Task_Flowchart)
# Preparing the Environment
- [Secure JupyterLab](#Secure_JL)
- [Permit root Login for SSH](#Permit_Root)
- [Update Global Settings](#Global)
- [SMTP Configuration (Optional)](#SMTP_Config)
# Depot Maintence
- [Resetting the Depot](#Depot_Reset)
- [Enable NFS service on Depot](#Enable_NFS)
- [Photon OS Update](#OS_Update)
    - [Python and JupyterLab Update](#JupyterLab_Update)
- [Setup Depot Self-Signing Certificate](#Certificate)
    - [Restoring saved Depot Self-Signing Certificate](#Certificate_Restore)
- [Configure NGINX Alias for legacy VCF 5.# (testing)](#Alias)
- [Update Network Routing on Depot](#Route)
    - [Reset to Default Network Route](#Reset_Route)
- [Upload Depot Certificate onto SDDC Manager](#UploadCert)
- [Upload Depot Certificate onto vCenter as Trusted Root](#UploadCert_VC)
- [Downloading VCF 5.2.x binaries to local depot (External)](#DL5_Ext)
    - [Download VCF 5.2.x vSAN HCL, Compatibility Matrix, and full Manifest](#DL5_Aux)
    - [Download VCF 5.2.x Bundles and correspond Bundle Manifests](#DL5_Bundles)
    - [Update downloaded bundles index](#DL5_Index)
- [Downloading VCF 9.x binaries to local depot (External)](#DL_Ext)
    - [List all binaries for a specific VCF version installation or upgrade](#DL_VCF_Installer)
    - [Download all binaries needed by the VCF Installer for the specified VCF releases](#DL_VCF_Installer)
    - [Download the ESXi binaries required for Holodeck](#DL_ESX)
    - [Download ALL binaries for specified VCF releases](#DL_ESX)
    - [Additional binaries/components for VCF 9.x and 9.1.x.](#DL_ADD)
    - [Additional binaries/components for VCF 9.1.x](#DL_ADD2)
    - [Upgrade managed by SDDC](#UG_SDDC)
    - [Upgrade managed by VRSLCM](#UG_VRSLCM)
    - [Upgrade managed by VCF Fleet LCM](#UG_VCFLCM)
- [Update vSphere Lifecycle Download (vLCM) source(s)](#vLCM)
    - [Update Lifecycle Download Source(s) on vCenter](#Update_vLCM)
    - [Initiate Sync Updates](#Sync_vLCM)
- [VMware UMDS (Update Manager Download Service)](#UMDS)
    - [Configure NGINX web server for UMDS](#NGINX_UMDS)
    - [Setup UMDS Self-Signing Certificate](#UMDS_Certificate)
        - [Restoring saved UMDS Self-Signing Certificate](#UMDS_Certificate_Restore)
    - [Upload UMDS Certificate onto vCenter as Trusted Root (Optional)](#UploadCert_UMDS)
    - [Configure VMware UMDS 8](#UMDS8_Config)
    - [Download UMDS 8 updates](#UMDS8_Update)
    - [Switching Patch Download Source to UMDS](#Switch_Depot)
    - [Configure VMware UMDS 9](#UMDS_Config)
    - [Download UMDS 9 updates](#UMDS_Update)
    - [Schedule UMDS updates](#UMDS_Schedule)
- [Setting Permissions](#Set_Perms)
- [Stage VCF 5.2.x binaries onto HoloRouter](#Stage_5)
- [Stage VCF 9.x binaries onto HoloRouter](#Stage_9)
- [Holodeck 9.0.2 Deployment Note](#Holodeck_Note)
- [Delete Depot Certificate from SDDC Manager](#DeleteCert)
- [Delete Depot Certificate from vCenter](#DeleteCert_VC)
- [Delete UMDS Certificate from vCenter](#DeleteCert_UMDS)
- [Expanding the Filesystem](#Expand_FS)
- [Changing the Basic Auth Credentials](#BasicAuth)

<a id="Why_JL"></a>
## Why we use Jupyter Notebooks

Juypter notebooks provide the ability for us to document specific tasks. However, that is just scratching the surface of what you can do in a jupyter notebook.

One of the most common features that are used here is the ability to run scripts from the notebook. This helps us to automate certain tasks and make completing them easier for you.

There are also other features that you can use to greate diagrams, real-time graphs, and so on. You should feel free to explore some of the things that you can do.

Lastly, using Jupyter notebooks helps us to share the information with others! 

[Return to Top of Page](#Home)

<a id="Using_JL"></a>
## How to use the Jupyter Notebook

[Return to Top of Page](#Home)

<a id="Sharing_JL"></a>
## Providing Feedback and Sharing




[Return to Top of Page](#Home)

<a id="Task_Flowchart"></a>
## Task Flowchart

The following outlines the order of operations you will follow to prepare the depot for your environment.

```mermaid
---
title: Order of Operations
---
flowchart TD
  A([Start]) --> B{Using Holodeck?}
  B -->|Yes| C[Install Holorouter]
  B -->|No| D[Secure Depot]
  D --> E[Get Download Token]
  E --> F[Download Bits]
  F --> G[Party Time!]
  C --> E
  

[Return to Top of Page](#Home)

<a id="Secure_JL"></a>
## Secure JupyterLab

By default, the Jupyter Labs server that runs on this depot is open to everyone. This means that anyone who knows the URL used to access the Jupyter server will be able to access the Jupyter notebooks. In a lab environment, this might not be a concern for you. However, as people would have the capacity to execute code in a Jupyter notebook this could expose the system to attack.

If desired, you can implement a password that would need to be entered in order to access the Jupyter server.

In order to do this, you would need to execute the following command:

<span style="color:red; font-weight:bold">/opt/vmware/env/bin/jupyter-lab password</font>

<div class="alert alert-block alert-danger">
<b>Danger:</b> DO NOT FORGET THE PASSWORD. Use with caution!
</div>

[Return to Top of Page](#Home)

<a id="Permit_Root"></a>
## Permit root Login for SSH
By default, the depot appliance follows some of the default security recommendations. This includes disabling the ability for root to directly SSH to the appliance. 
In a lab environment, you may wish to allow this to make your workflow easier.

**This is not recommended in a production environment.**

<div class="alert alert-block alert-danger">
<b>Danger:</b> This task will permit SSH root login on the Depot server.  Proceed with caution.
</div>

In [ ]:
if grep -q -- "PermitRootLogin no" "/etc/ssh/sshd_config"; then
    echo "SSH Root Login prohibited.  Setting SSH to allow Root Login."
    sudo sed -i 's/^PermitRootLogin no/PermitRootLogin yes/' /etc/ssh/sshd_config
else
    echo "SSH Root Login already permitted."
fi
sudo systemctl restart sshd
echo "SSH service restarted." 

[Return to Top of Page](#Home)

<a id="Auto_Token"></a>
## Auto Download Token / Activation Code Renewal
The Broadcom Internal Employee Download Token (and Activation Code) automatically expires every 48 hours and requires manual request/renewal. The cell below enables automated background request and token renewal, while updating the local token file used by various automation scripts across the ODA JupyterLab notebooks.

<div class="alert alert-block alert-danger">
<b>CAUTION:</b> THIS FUNCTIONALITY IS AVAILABLE EXCLUSIVE AND SHOULD ONLY BE ENABLE BY BROADCOM INTERNAL EMPLOYEES!
</div>

In [ ]:
### Run this for TOKEN use only ##
### Further modifications to the scripts will be necessary to accommodate the Token use case, 
### which is in the process of being retired
#
# Note: For token use case, replace "--depot-download-activation-code-file=/root/activation_code.txt" with 
# "--depot-download-token-file=/root/download_token.txt"
#
mkdir -p /root/cron
rm -f /root/cron/token.sh
echo "/opt/vmware/env/bin/jupyter nbconvert --to notebook --execute --inplace /var/notebooks/Token.ipynb" > /root/cron/token.sh
# jupyter nbconvert --clear-output --inplace notebook.ipynb
chmod 744 /root/cron/*
#
cat << EOF | sudo tee /etc/cron.d/token >/dev/null
SHELL=/bin/bash
PATH=/sbin:/bin:/usr/sbin:/usr/bin
# Job 1 - Token refresh every 48 hours at 11:00 PM
0 23 */2 * * root /root/cron/token.sh >> /var/log/token-refresh.log 2>&1 | printf "Subject:$Email_Subject\n\nAuto token refresh is completed." | msmtp $Email_To
EOF
sudo chmod 644 /etc/cron.d/token
echo "Token refresh cron jobs added."

In [ ]:
### Run this for Activation Code use only ###
#
mkdir -p /root/cron
rm -f /root/cron/activation.sh
echo "/opt/vmware/env/bin/jupyter nbconvert --to notebook --execute --inplace /var/notebooks/Activation.ipynb" > /root/cron/activation.sh
# jupyter nbconvert --clear-output --inplace notebook.ipynb
chmod 744 /root/cron/*
#
cat << EOF | sudo tee /etc/cron.d/token >/dev/null
SHELL=/bin/bash
PATH=/sbin:/bin:/usr/sbin:/usr/bin
# Job 1 - Token refresh every 48 hours at 11:00 PM
0 23 */2 * * root /root/cron/activation.sh >> /var/log/activation-refresh.log 2>&1 | printf "Subject:$Email_Subject\n\nAuto Activation Code refresh is completed." | msmtp $Email_To
EOF
sudo chmod 644 /etc/cron.d/activation
echo "Activation Code refresh cron jobs added."

[Return to Top of Page](#Home)

<a id="Global"></a>
## Update Global Settings
This section allows you to define values for a number of variables that are used throughout this notebook. This includes defining the download token, the location for the Broadcom depot, information needed to manage the certificates, Holodeck-specific values, and various other global variables. 

You should review the sample information here, and update accordingly. Once updated, go ahead and run the code cell in order for all the variables to be defined.

If you need to update the variables, simply come back here, update as needed, and then re-run the cell.

In [6]:
# Uncomment one of the below values to match your timezone
MyTimeZone="America/New_York" 
#MyTimeZone="America/Chicago"
#MyTimeZone="America/Denver"
#MyTimeZone="America/Los_Angeles"
#MyTimeZone="America/Anchorage"
#MyTimeZone="America/Adak"
#MyTimeZone="Europe/London"
#MyTimeZone="Europe/Paris"
#MyTimeZone="Europe/Berlin"
#MyTimeZone="Europe/Moscow"
#MyTimeZone="Asia/Tokyo"
#MyTimeZone="Asia/Singapore"
#MyTimeZone="Asia/Kolkata"
#MyTimeZone="Asia/Dubai"

DOWNLOAD_TOKEN=tNH3HwVqh3T9dQXOMwXD87FF1FTd9NQf
echo ${DOWNLOAD_TOKEN} > /root/download_token.txt
#
# Enable for those who need to use the depot password
# depotUser_Password=' '
# echo ${depotUser_Password} > /root/depotUserPassword
# chmod 400 /root/depotUserPassword
LCM_HOST=dl-pstg.broadcom.com # Set which Broadcom depot to use - "dl.broadcom.com" for production
LCM_HOST2=dl-vcfstg.broadcom.com
#
# Self-Signed Depot Certificate info
SSL_Country=US
SSL_State=OH
SSL_Locality='Concord Township'
SSL_OrgName=Tataoui
SSL_OrgUnit=IT
SSL_CommonName=depot.tataoui.com
SSL_eMail=dwchan@gmail.com
SSL_SANName=$SSL_CommonName
SSL_SANIP=$(/usr/sbin/ip -o -4 addr show dev eth0 | awk '{print $4}' | cut -d/ -f1)
#
# Self-Signed UMDS Certificate info
UMDS_SSL_CommonName=umds.tataoui.com
UMDS_SSL_SANName=$UMDS_SSL_CommonName
#
UMDS_Tarball='VMware-UMDS-8.0.3.00800-15560290.tar.gz' # Based on vCenter 8.0 Update 3i Build 25197330
UMDS8_PatchStore="/var/www/build/umds8-patch-store"
if [ ! -d "$UMDS8_PatchStore" ]; then
    echo "VMware UMDS 8 Patch Repository missing, creating now."
    mkdir -p $UMDS8_PatchStore
    chmod 750 $UMDS8_PatchStore
else
    echo "VMware UMDS 8 Patch Repository found at $UMDS8_PatchStore"
fi
UMDS8_PatchStore_Export="/var/www/build/umds8"
if [ ! -d "$UMDS8_PatchStore_Export" ]; then
    mkdir -p $UMDS8_PatchStore_Export
    chmod 750 $UMDS8_PatchStore_Export
    echo "VMware UMDS 8 Patch Export Store missing, creating now."
else
    echo "VMware UMDS 8 Patch Export Store found at $UMDS8_PatchStore_Export"
fi
#
UMDS_PatchStore="/var/www/build/umds-patch-store"
if [ ! -d "$UMDS_PatchStore" ]; then
    mkdir -p $UMDS_PatchStore
    chmod 750 $UMDS_PatchStore
    echo "VMware UMDS 9 Patch Repository missing, creating now."
else
    echo "VMware UMDS 9 Patch Repository found at $UMDS_PatchStore"
fi
UMDS_PatchStore_Export="/var/www/build/umds"
if [ ! -d "$UMDS_PatchStore_Export" ]; then
    mkdir -p $UMDS_PatchStore_Export
    chmod 750 $UMDS_PatchStore_Export
    echo "VMware UMDS 9 Patch Export Store missing, creating now."
else
    echo "VMware UMDS 9 Patch Export Store found at $UMDS_PatchStore_Export"
fi
#
# VMware Holodeck info - May not apply to all environments and user cases
HoloRouter_Ext_IP=192.168.10.3
HoloRouter_FQDN=holorouter9.tataoui.com
HoloRouter_Pass='VMware123!'
HoloDeck_Network='10.1.0.0'
HoloDeck_Subnet='255.255.240.0'
HoloDeck_GW='10.1.0.1'
if ssh-keygen -F "$HoloRouter_FQDN" > /dev/null; then
  echo "Host '$HoloRouter_FQDN' is already in known_hosts.  Nothing to do."
else
  echo "Host $HoloRouter_FQDN is missing. Adding it now..."
  ssh-keyscan -H "$HoloRouter_FQDN" >> ~/.ssh/known_hosts
fi
VCF5_versions=("5.2.0" "5.2.1" "5.2.2" "5.2.3")
VCF5_ESX_versions=("8.0U3-" "8.0U3b" "8.0U3g")
# For VCF 5.2.x deployment, the required binaries must be uploaded onto the depot manually - /var/www/build/Legacy
# VCF 5.2
# - VMware-Cloud-Builder-5.2.0.0-24108943_OVF10.ova
# - VMware-VMvisor-Installer-8.0U3-24022510.x86_64.iso
# VCF 5.2.1
# - VMware-Cloud-Builder-5.2.1.0-24307856_OVF10.ova
# - VMware-VMvisor-Installer-8.0U3b-24280767.x86_64.iso
# VCF 5.2.2
# - VMware-Cloud-Builder-5.2.2.0-24936865_OVF10.ova
# - VMware-VMvisor-Installer-8.0U3g-24859861.x86_64.iso
# VCF 5.2.3
# - VMware-Cloud-Builder-5.2.3.0-25219033_OVF10
# - VMware-VMvisor-Installer-8.0U3i-25205845.x86_64.iso
#
VCF9_versions=("9.0.0.0" "9.0.1.0" "9.0.2.0")
VDT_LOGFILE="/root/vdt/log/vdt_download.log"
VCF_versions=("9.0.0.0" "9.0.1.0" "9.0.2.0" "9.1.0")
VCF_components=(
    VCENTER
    SDDC_MANAGER_VCF
    NSX_T_MANAGER
    ESX_HOST
    VRSLCM
    VRA
    VROPS
    VRLI
    VRNI
    VSAN_OSA_WITNESS
    VSAN_ESA_WITNESS
    VSAN_FILE_SERVICES
    VMTOOLS
    VCFDT
    VCF_OPS_CLOUD_PROXY
    VIDB
    HCX
    VMRC
    VRO
    VSP
    DEPOT_SERVICE
    VCF_LICENSE_SERVER
    TELEMETRY_ACCEPTOR
    VCF_FLEET_LCM
    VCF_SDDC_LCM
    VCF_CONSUMPTION_CLI
    VCF_CONSUMPTION_CLI_PLUGINS
    VCFMS_METRICS_STORE
    VCF_SERVICE_VCD_MIGRATION_BACKEND
    VCF_SALT
    VCF_SALT_RAAS
    VCF_OBSERVABILITY_DATA_PLATFORM
)
#
# VCF_VERSION=9.0.2 # Add comment on here before release
VCF_VERSION=9.1.0 # Add comment on here before release
VCF_VERSION_LEGACY=5.2.3.0
#
# sddcManagerFQDN=sddc-manager.vcf.sddc.lab
# sddcManagerUser=administrator@vsphere.local
# sddcManagerFQDN=vcfinstaller.tataoui.com #Standalone VCF Installer
#
# Physical vCenter
vCenterFQDN="vc01.tataoui.com"
vCenterUser="administrator@vsphere.local"
vCenterPassword='VMware123!'
#
# Physical SDDC Manager
# sddcManagerFQDN=vcf01-m01-sddc.tataoui.com # VCF 5.x SDDC Manager
# sddcManagerUser=admin@local #Standalone VCF Installer / VCF 5.x SDDC Manager admin username
# sddcManagerPass='VMware123!VMware123!'
#
# Holodeck SDDC Manager - Site a
sddcManagerFQDN="sddcmanager-a.site-a.vcf.lab"
sddcManagerUser="administrator@vsphere.local"
sddcManagerPass='VMware123!VMware123!'
#
# Physical VCF Installer
vcfInstallerFQDN="vcfinstaller.tataoui.com"
vcfInstallerUser="admin@local"
vcfInstallerPass='VMware123!VMware123!'
#
# Holodeck VCF Installer - Site a
# vcfInstallerFQDN=vcfinstaller-a.site-a.vcf.lab
# vcfInstallerUser=admin@local
# vcfInstallerPass='VMware123!VMware123!'
#
# Configure the following SMTP settings to enable email notifications. 
Email_From='dwchan69@gmail.com'
Email_To='dwchan69@gmail.com'
Email_Subject='VCF Offline Depot Appliance Notification'
SMTP_Username='dwchan69@gmail.com'
SMTP_Password='juoxcmbxjrkjfwzw'
#
echo "Global setting variables updated."

VMware UMDS 8 Patch Repository found at /var/www/build/umds8-patch-store
VMware UMDS 8 Patch Export Store found at /var/www/build/umds8
VMware UMDS 9 Patch Repository found at /var/www/build/umds-patch-store
VMware UMDS 9 Patch Export Store found at /var/www/build/umds
Host 'holorouter9.tataoui.com' is already in known_hosts.  Nothing to do.
Global setting variables updated.


[Return to Top of Page](#Home)

<a id="Global"></a>
## Read in Global Variables Setting
Executing the following cell will read in all the Global Variables that you have saved and make them available within this notebook.
You will need to do this each time you open this notebook.  You can also re-run this if you have made changes to the Global Variables page.

In [35]:
source /root/env_vars.txt
echo "Global setting variables are read and loaded from file."

<a id="SMTP_Config"></a>
## SMTP Configuration (Optional)
Configure SMTP settings on the Offline Depot Appliance (ODA) to enable email notifications for scheduled tasks or synchronization jobs that may take an extended time to complete.

In [ ]:
tdnf install -y msmtp
#
touch /var/log/msmtp.log
cat << EOF > /etc/msmtprc
defaults
auth on
tls on
tls_starttls   on
tls_trust_file /etc/pki/tls/certs/ca-bundle.crt
logfile /var/log/msmtp.log
# Gmail Account Configuration
account Gmail
host smtp.gmail.com
port 587
from $ODA_Email_From
user $ODA_SMTP_Username
password $ODA_SMTP_Password
#
account default: Gmail
EOF
chmod 600 /etc/msmtprc
ln -sf /usr/bin/msmtp /usr/sbin/sendmail
ln -sf /usr/bin/msmtp /usr/bin/mail
cat << EOF > /etc/aliases
default: $ODA_Email_To
EOF
echo "Email notification and account configured."

[Return to Top of Page](#Home)

<a id="Depot_Reset"></a>
## Resetting the Depot
This task will delete all the VCF binaries on the depot. This is commonly done by people who are testing different versions of VCF or in the event that you just want to start over, yet still keep the information (like this Jupyter notebook) that you already have customized.

After doing this, you can download the binaries again. Note that once you do this, you should ensure that you have set the permissions appropriately.

Also note, this will not delete the VCF Product docs that are available on the depot.

<div class="alert alert-block alert-danger">
<b>Danger:</b> This will delete ALL binaries hosted by this depot. Use with caution!
</div>

In [ ]:
cd /var/www/build
find . -mindepth 1 -maxdepth 1 -type d ! -name "docs" -exec rm -rf {} +
echo "Depot Reset Complete"

[Return to Top of Page](#Home)

<a id="Enable_NFS"></a>
## Enable NFS service on Offline Depot (Optional)

Enable NFS service on the Offline Depot Appliance (ODA) to host Aria Suite Lifecycle Manager product and patch binaries for the VMware Cloud Foundation (VCF) 5.2.x environment.

In [ ]:
vRLCM_RepoDir="/var/www/build/VRLCM"
#
if ! tdnf list installed nfs-utils &>/dev/null; then
    echo "'nfs-utils' package not found."
    sudo tdnf install -y nfs-utils
    echo "'nfs-utils' package installed."
else
    echo "Package 'nfs-utils' is already installed."
fi

if [ ! -d "$vRLCM_RepoDir" ]; then
    mkdir "$vRLCM_RepoDir"
    chmod 755 $vRLCM_RepoDir
    echo "Aria Suite Lifecycle Product Binaries Repo directory created on depot."
else
    echo "Aria Suite Lifecycle Product Binaries Repo directory already existed at $vRLCM_RepoDir"
fi

echo "$vRLCM_RepoDir *(rw,sync,no_root_squash,no_subtree_check)" > /etc/exports
echo "MOUNTD_PORT=2049, LOCKD_TCPPORT=2049, RQUOTAD_PORT=874" > /etc/sysconfig/nfs
/usr/sbin/exportfs -arv
systemctl enable nfs-server.service
systemctl start nfs-server.service
echo "NFS service enabled"
#
/usr/sbin/iptables -A INPUT -p tcp --dport 2049 -j ACCEPT
/usr/sbin/iptables -A INPUT -p udp --dport 2049 -j ACCEPT
/usr/sbin/iptables-save > /etc/systemd/scripts/ip4save
systemctl restart iptables
echo "NFS service/network configuration completed." 

[Return to Top of Page](#Home)

<a id="OS_Update"></a>
## Photon OS Update

Execute the following cell as needed to update the appliance Operating System (VMware Photon OS)

In [ ]:
tdnf update --assumeyes
tdnf clean all
tdnf makecache
echo "VMware Photon OS updated." 

[Return to Top of Page](#Home)

<a id="JupyterLab_Update"></a>
### Python and JupyterLab Update
<div class="alert alert-block alert-info">
Update Python and JupyterLab as required. Be sure to snapshot the virtual machine (ODA) beforehand and proceed with caution.
</div>

In [ ]:
source /opt/vmware/env/bin/activate
pip install --upgrade jupyterlab
echo "JupyterLab version updated." 

[Return to Top of Page](#Home)

<a id="Certificate"></a>
## Setup Depot Self-Signing Certificate

Generate a self‑signed certificate, create a backup, and update the NGINX configuration to enable HTTPS for the depot services.

In [ ]:
if [[ -e "/etc/nginx/ssl/$HOSTNAME.crt" ]] then
    echo "Looks like a self-signing certificate - '$HOSTNAME.crt' already exists at /etc/nginx/ssl."
    echo "If you think this is an error or you want to generate a new (or expired) certificate, please manually delete the certificate inside folder '/etc/nginx/ssl/' and try again."
else
    openssl req -x509 -nodes -days 365 -newkey rsa:2048 \
    -keyout /etc/nginx/ssl/$HOSTNAME.key -out /etc/nginx/ssl/$HOSTNAME.crt \
    -subj "/C=$SSL_Country/ST=$SSL_State/L=$SSL_Locality/O=$SSL_OrgName/OU=$SSL_OrgUnit/CN=$SSL_CommonName/emailAddress=$SSL_eMail" \
    -addext "subjectAltName=DNS:$SSL_SANName,IP:$SSL_SANIP" \
    -addext "keyUsage=critical,digitalSignature,keyCertSign" \
    -addext "extendedKeyUsage=serverAuth" \
    -addext "basicConstraints=critical,CA:TRUE"
    sed -i -e "s|server_name  localhost|server_name  $HOSTNAME|" /etc/nginx/nginx.conf
    sed -i -e "s|/root/certs/localhost.crt|/etc/nginx/ssl/$HOSTNAME.crt|" /etc/nginx/nginx.conf
    sed -i -e "s|/root/certs/localhost.key|/etc/nginx/ssl/$HOSTNAME.key|" /etc/nginx/nginx.conf
    echo "New self-signing certificates generated for offline depot web service (NGINX)."
    sudo systemctl restart nginx
    echo "NGINX service restarted."
    timestamp=$(date -u +"%Y-%m-%d_%H-%M")
    tar -czPf /root/depotCerts_${timestamp}.tar.gz /etc/nginx/ssl/
    echo "Depot Certificates are backed up on '/root' as 'depotCerts_${timestamp}.tar.gz'.  Please download and save it to a secure location in case a restore is required."
fi

[Return to Top of Page](#Home)

<a id="Certificate_Restore"></a>
### Restore previously saved Depot Self-Signing Certificate (For rebuild)
<div class="alert alert-block alert-danger">
Restore a previously saved self‑signed depot certificate when performing a system recovery. Upload the saved certificate tarball (filename.tar.gz) to /root, and update the certFilename variable accordingly. Proceed with caution!!
</div>

In [ ]:
certFilename="depotCerts_2026-05-05_01-10.tar.gz" # Update restore certifcate filename (time-stamp) accordingly i.e. depotCerts_2026-04-24_01-22.tar.gz
if [ -d "/etc/nginx/ssl" ]; then
    echo "NGINX SSL directory already exists."
else
    echo "NGINX SSL directory does not exist. Creating directory."
    mkdir -p /etc/nginx/ssl
    chmod 700 /etc/nginx/ssl
fi
if [ -f "/root/$certFilename" ]; then
    echo "Save Depot certificate tarball found."
    tar -xzvpf /root/$certFilename -C /
    echo "Save Depot certificates (NGINX) are restored."
    sed -i -e "s|server_name  localhost|server_name  $HOSTNAME|" /etc/nginx/nginx.conf
    sed -i -e "s|/root/certs/localhost.crt|/etc/nginx/ssl/$HOSTNAME.crt|" /etc/nginx/nginx.conf
    sed -i -e "s|/root/certs/localhost.key|/etc/nginx/ssl/$HOSTNAME.key|" /etc/nginx/nginx.conf
    echo "NGINX configuration updated."
    sudo systemctl restart nginx
    echo "NGINX service restarted." 
else
    echo "No saved Depot certificate tarball was located."
fi

[Return to Top of Page](#Home)

<a id="Alias"></a>
## Configure NGINX Alias for VCF 5.# (not needed for VCF 5.2 and above)
Create alias directives to support VCF 5.0 and 5.1 SDDC Manager when using earlier versions of the VMware Offline Bundle Transfer Utility (OBTU).

In [ ]:
sed -i -e "s|/404.html;|/404.html; \n
\tlocation /PROD2 {\n \
\t\talias /var/www/build/PROD;\n \
\t\tindex index.html index.htm;\n \
\t\tautoindex on;\n \
\t\tauth_basic 'Restricted Access';\n \
\t\tauth_basic_user_file /etc/nginx/.htpasswd;\n \
\t}\n \
\tlocation /PROD2/evo/vmw {\n \
\t\talias /var/www/build/PROD/COMP/SDDC_MANAGER_VCF;\n \
\t\tindex index.html index.htm;\n \
\t\tautoindex on;\n \
\t\tauth_basic 'Restricted Access';\n \
\t\tauth_basic_user_file /etc/nginx/.htpasswd;\n \
\t}\n \
\tlocation /PROD2/evo/vmw/lcm {\n \
\t\talias /var/www/build/PROD/metadata;\n \
\t\tindex index.html index.htm;\n \
\t\tautoindex on;\n \
\t\tauth_basic 'Restricted Access';\n \
\t\tauth_basic_user_file /etc/nginx/.htpasswd;\n \
\t}\n|" /etc/nginx/nginx.conf

[Return to Top of Page](#Home)

<a id="Route"></a>
## Update Network Routing on Depot
Adding static route to HoloDeck nested environment onto Offline Depot and your physical workstation network environment.

In [6]:
# sed -i '/Route/{N;N;d;}' /etc/systemd/network/50-dhcp-en.network
mask2cidr() {
    local nbits=0
    IFS=.
    for octet in $1; do
        case $octet in
            255) let nbits+=8;;
            254) let nbits+=7; break;;
            252) let nbits+=6; break;;
            248) let nbits+=5; break;;
            240) let nbits+=4; break;;
            224) let nbits+=3; break;;
            192) let nbits+=2; break;;
            128) let nbits+=1; break;;
            0)   break;;
            *)   echo "Invalid octet: $octet"; return 1;;
        esac
    done
    echo "$nbits"
}
ping -c 1 -W 1 $HoloDeck_GW &> /dev/null
if [ $? -eq 0 ]; then
  echo "Route to HoloDeck network gateway $HoloDeck_GW exists."
else
    echo "Route to HoloDeck network $HoloDeck_GW does not exist.  Adding a static route to HoloDeck environment on Offline Depot."
    
    if ! grep -qxF "[Route]" /etc/systemd/network/50-dhcp-en.network; then
        echo "[Route]" >> /etc/systemd/network/50-dhcp-en.network
        echo "Gateway=$HoloRouter_Ext_IP" >> /etc/systemd/network/50-dhcp-en.network
        echo "Destination=$HoloDeck_Network/$(mask2cidr "$HoloDeck_Subnet")" >> /etc/systemd/network/50-dhcp-en.network
        systemctl restart systemd-networkd
        echo "[Route] header added to /etc/systemd/network/50-dhcp-en.network."
        echo "New persistence static route to HoloDeck network '$HoloDeck_Network' added onto Offline Depot."
        # for non-persistence use case, add this manual from the terminal
        # route add -net 10.1.0.0 netmask 255.255.240.0 gw $HoloRouter_Ext_IP dev eth0
    else
        echo "[Route] header and statis route already exist."
    fi
    echo "Run the following cell and copy its output to your local workstation with administrator privileges"
    echo "if you want to access the Holodeck nested environment directly, without using the Holodeck Webtop."
    echo "route add $HoloDeck_Network mask $HoloDeck_Subnet $HoloRouter_Ext_IP"
fi
ping -c 1 -W 1 $sddcManagerFQDN &> /dev/null
if [ $? -eq 0 ]; then
    echo "SDDC manager - '$sddcManagerFQDN' is reachable."
else
    echo "SDDC manager - '$sddcManagerFQDN' is not reachable."
    # exit 1
fi

Route to HoloDeck network 10.1.0.1 does not exist.  Adding a static route to HoloDeck environment on Offline Depot.
[Route] header added to /etc/systemd/network/50-dhcp-en.network.
New persistence static route to HoloDeck network '10.1.0.0' added onto Offline Depot.
Copy and run the following command on your local workstation with administrator rights (optional)
route add 10.1.0.0 mask 255.255.240.0 192.168.10.3
SDDC manager - 'sddcmanager-a.site-a.vcf.lab' is not reachable.


[Return to Top of Page](#Home)

<a id="Reset_Route"></a>
### Reset to Default Network Route
Remove the Holodeck network routing entries from the ODA and restore the routing table to its default configuration.
<div class="alert alert-block alert-info">
<b>Note:</b>  This will remove previous static route entry added by the script.
</div>

In [ ]:
sed -i '/Route/,$d' /etc/systemd/network/50-dhcp-en.network
systemctl restart systemd-networkd

[Return to Top of Page](#Home)

<a id="UploadCert"></a>
## Upload Depot Certificate onto SDDC Manager
Adding the newly created self-signing certificate (depot) to the VCF SDDC Manager to enable HTTPS communication without errors.

In [ ]:
HTTP_Code_Token=$(curl -sk -w "%{http_code}" -o /tmp/token_response.json -X POST https://$sddcManagerFQDN/v1/tokens \
    -H 'Content-Type:application/json' \
    -d '{"username": "'$sddcManagerUser'","password": "'$sddcManagerPass'"}')
echo "Access Token HTTP status: $HTTP_Code_Token"
AccessToken=$(jq -r '.accessToken' /tmp/token_response.json)
if [[ $HTTP_Code_Token != "200" || -z $AccessToken || $AccessToken == "null" ]]; then
    echo "Failed to retrieve Access Token (HTTP $HTTP_Code_Token)."
    cat /tmp/token_response.json
    exit 1
else
    echo "Access Token acquired."
fi

certFile=/etc/nginx/ssl/$HOSTNAME.crt
jq -nc --rawfile cert "$certFile" '{certificate:$cert, certificateUsageType:"TRUSTED_FOR_OUTBOUND"}' > /tmp/certificate.json

HTTP_Code_Cert=$(curl -sk -w "%{http_code}" -o /tmp/certResult.json -X POST "https://$sddcManagerFQDN/v1/sddc-manager/trusted-certificates" \
    -H "Authorization: Bearer $AccessToken" -H "Content-Type: application/json" --data-binary @/tmp/certificate.json)
    
echo "Upload HTTP status: $HTTP_Code_Cert"
jq . /tmp/certResult.json || cat /tmp/certResult.json

if [[ "$HTTP_Code_Cert" =~ ^2 ]]; then
    echo "Certificate uploaded successfully."
else
    echo "Certificate upload failed (HTTP $HTTP_Code_Cert)"
    exit 1
fi

[Return to Top of Page](#Home)

<a id="UploadCert_VC"></a>
## Upload Depot Certificate onto vCenter as Trusted Root
Adding the newly created self-signing certificate (depot) to vCenter as a Trusted Root to enable HTTPS communication with vCenter Lifecycle Manager without errors.

In [ ]:
CertFile="/etc/nginx/ssl/$HOSTNAME.crt"

SessionID=$(curl -k -s -X POST -u "$vCenterUser:$vCenterPassword" "https://$vCenterFQDN/api/session" -H 'Accept: application/json' | tr -d '"')
if [ -n "$SessionID" ]; then
    echo "Successfully obtained Session ID: $SessionID"
else
    echo "Authentication failed.  Failed to obtain Session ID."
fi
CertData=$(sed -E ':a;N;$!ba;s/\r{0,1}\n/\\n/g' "$CertFile")

Payload=$(cat <<EOF
{
  "cert_chain": {
    "cert_chain": [ "$CertData" ]
  }
}
EOF
)
Response=$(curl -k -s -X POST "https://$vCenterFQDN/api/vcenter/certificate-management/vcenter/trusted-root-chains" \
    -H "vmware-api-session-id: $SessionID" -H "Content-Type: application/json" -d "$Payload")
if [ "$Response" == "400" ] || [ "$Response" == "401" ]; then

    echo "Failed to add certificate. HTTP Status: $Response"
else
    echo "Root certificate successfully added. - $Response"
fi

[Return to Top of Page](#Home)

<a id="DL5_Ext"></a>
## Downloading VCF 5.2.x binaries to local depot

The following cells use the VCF Bundle Transfer Utility (OBTU) to download the required VCF 5.x installation binaries, patches, and component updates to the local offline depot, supporting both convenience and fully air‑gapped environments.
Begin by updating the Broadcom external/public repository URL for your VCF 5.2.x downloads.

In [ ]:
cd /root
mkdir /root/vdt
cp /root/vcf-download-tool-9.1.0.0.25371089.tar.gz /root/vdt
cd /root/vdt
tar -zxvf vcf-download-tool-9.1.0.0.25371089.tar.gz
cd bin
chmod +x *

CONFIG="/root/vdt2/conf/application-prod.properties"

if [ ! -f "$CONFIG" ]; then
    yes y | /root/vdt2/bin/vcf-download-tool binaries list \
      --depot-download-activation-code-file=/root/activation_code.txt \
      --vcf-version=5.2.3 \
      --type=INSTALL
else
    /root/vdt2/bin/vcf-download-tool binaries list \
      --depot-download-activation-code-file=/root/activation_code.txt \
      --vcf-version=5.2.3 \
      --type=INSTALL
fi

In [ ]:
# Legacy VDT 9.0.2 requirement
# sed -i -e "s|lcm.depot.adapter.host=.*|lcm.depot.adapter.host=$LCM_HOST|" /root/vdt/conf/application-prod.properties
#
# Using embedded Token base authenication
# sed -i -e "s|lcm.depot.adapter.remote.v2.rootDir=.*|lcm.depot.adapter.remote.v2.rootDir=/$DOWNLOAD_TOKEN/PROD|" /root/vdt/conf/application-prod.properties
# echo "Setting the latest Broadcom Repository URL for VCF Bundle Transfer Utility."

In [ ]:
sed -i -e "s|lcm.depot.adapter.host=.*|lcm.depot.adapter.host=$LCM_HOST2|" /root/vdt/conf/application-prod.properties
sed -i -e "s|lcm.depot.adapter.enableBundleSignatureValidation=.*|lcm.depot.adapter.enableBundleSignatureValidation=false|" /root/vdt/conf/application-prod.properties
grep -qx "lcm.access_token.broadcom.authorization.server.url=https://eapi-gcpstg.broadcom.com/vcf/generateToken" /root/vdt/conf/application-prod.properties \
  || printf "\n%s\n" "lcm.access_token.broadcom.authorization.server.url=https://eapi-gcpstg.broadcom.com/vcf/generateToken" >> /root/vdt/conf/application-prod.properties
echo "Setting the latest Broadcom Repository URL for VCF Bundle Transfer Utility."
# How to fix the token
mkdir -p /etc/security/token
chmod 700 /etc/security/token
echo "token=$DOWNLOAD_TOKEN" > /etc/security/token/token.properties
chmod 600 /etc/security/token/token.properties
echo "Setting the required token file for Bundle Transfer Utility 9.1 (lcm-bundle-transfer-util)."

In [ ]:
# Other working examples
# ./lcm-bundle-transfer-util --listBundles --depotDownloadToken Ys8lwVoEoIDrp1474b8d8BBFar9tlRkf --productVersion 5.2.3.0
# ./lcm-bundle-transfer-util --download --compatibilityMatrix --outputDirectory /var/www/build --du dc005149@broadcom.net
# ./lcm-bundle-transfer-util --download --outputDirectory /var/www/build --depotDownloadToken Ys8lwVoEoIDrp1474b8d8BBFar9tlRkf --bundle bundle-120815

In [ ]:
/root/vdt/bin/lcm-bundle-transfer-util --setUpOfflineDepot --offlineDepotRootDir /var/www/build --offlineDepotUrl http://192.168.10.9/ --depotDownloadActivationCodeFile /root/activation_code.txt --sourceVersion 5.2.3.0

In [ ]:
/root/vdt/bin/lcm-bundle-transfer-util --download --outputDirectory /var/www/build/PROD/COMP/SDDC_MANAGER_VCF --productVersion 5.2.3.0 --depotDownloadActivationCodeFile /root/activation_code.txt

In [ ]:
/root/vdt/bin//lcm-bundle-transfer-util --download --outputDirectory /var/www/build --depotDownloadActivationCodeFile /root/activation_code.txt --productVersion 5.2.3.0 --imageType INSTALL

[Return to Top of Page](#Home)

<a id="DL5_Aux"></a>
### Download VCF 5.2.x vSAN HCL, Compatibility Matrix, and full Manifest
Download critical metadata and compatibility files for VMware Cloud Foundation (VCF) 5.2.x

In [ ]:
#yes | /root/vdt/bin/lcm-bundle-transfer-util --download --vsanHclDownload --outputDirectory /var/www/build/PROD --depotDownloadActivationCodeFile /root/activation_code.txt
#yes | /root/vdt/bin/lcm-bundle-transfer-util --download --compatibilityMatrix --outputDirectory /var/www/build/PROD/COMP/SDDC_MANAGER_VCF --depotDownloadActivationCodeFile /root/activation_code.txt
#yes | /root/vdt/bin/lcm-bundle-transfer-util --download --manifestDownload --outputDirectory /var/www/build/PROD/COMP/SDDC_MANAGER_VCF --depotDownloadActivationCodeFile /root/activation_code.txt
#
# echo "Latest VCF $VCF_VERSION_LEGACY vSAN HCL, Compatibility Matrix, and Manifest downloaded.   Log is available for review at /root/vdt/log/obtu.log"

In [ ]:
yes | /root/vdt/bin/lcm-bundle-transfer-util --download --vsanHclDownload --outputDirectory /var/www/build/PROD --depotDownloadActivationCodeFile /root/activation_code.txt
yes | /root/vdt/bin/lcm-bundle-transfer-util --download --compatibilityMatrix --outputDirectory /var/www/build/PROD/COMP/SDDC_MANAGER_VCF --depotDownloadActivationCodeFile /root/activation_code.txt
yes | /root/vdt/bin/lcm-bundle-transfer-util --download --manifestDownload --outputDirectory /var/www/build/PROD/COMP/SDDC_MANAGER_VCF --depotDownloadActivationCodeFile /root/activation_code.txt
#
echo "Latest VCF $VCF_VERSION_LEGACY vSAN HCL, Compatibility Matrix, and Manifest downloaded.   Log is available for review at /root/vdt/log/obtu.log"

In [4]:
cat > /root/run_lcm.expect << 'EOF'
#!/usr/bin/expect -f
set timeout -1

set cmd ""
foreach arg $argv {
    append cmd "$arg "
}
puts "\n{DEBUG} argv = $argv"
puts "{DEBUG} cmd  = $cmd\n"
spawn bash -c "$cmd"
expect {
    {Please acknowledge that the above requirements are met and press } {
        send "Y\r"
        exp_continue
    }
    {bundles (Y/N)?:} {
        send "Y\r"
        exp_continue
    }
    {Do you want to enable CEIP?} {
        send "N\r"
        exp_continue
    }
    eof
}
EOF
chmod +x /root/run_lcm.expect

In [ ]:
run_lcm() {
  /usr/bin/expect /root/run_lcm.expect "$@"
}
start_ts=$(TZ="$MyTimeZone" date '+%Y-%m-%d %H:%M:%S %Z')
echo "[$start_ts] Starting download for VCF $VCF_VERSION_LEGACY vSAN HCL, Compatibility Matrix, and Manifest" | tee -a "$VDT_LOGFILE"

run_lcm /root/vdt/bin/lcm-bundle-transfer-util \
  --download \
  --vsanHclDownload \
  --outputDirectory /var/www/build/PROD \
  --depotDownloadActivationCodeFile /root/activation_code.txt 2>&1 | tee /root/prompt.txt

run_lcm /root/vdt/bin/lcm-bundle-transfer-util \
  --download \
  --compatibilityMatrix \
  --outputDirectory /var/www/build/PROD/COMP/SDDC_MANAGER_VCF \
  --depotDownloadActivationCodeFile /root/activation_code.txt 
  
run_lcm /root/vdt/bin/lcm-bundle-transfer-util \
  --download \
  --manifestDownload \
  --outputDirectory /var/www/build/PROD/COMP/SDDC_MANAGER_VCF \
  --depotDownloadActivationCodeFile /root/activation_code.txt 

end_ts=$(TZ="$MyTimeZone" date '+%Y-%m-%d %H:%M:%S %Z')
echo "[$end_ts] Latest VCF $VCF_VERSION_LEGACY vSAN HCL, Compatibility Matrix, and Manifest download completed.  Log is available for review at /root/vdt/log/obtu.log" | tee -a "$VDT_LOGFILE"
 
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nLCM Bundle Transfer Util download activities are completed." | msmtp $Email_To
fi

[Return to Top of Page](#Home)

<a id="DL5_Bundles"></a>
### Download VCF 5.2.x Bundles and correspond Bundle Manifests
Download the deployment, update, and patch packages, along with the JSON‑formatted metadata files required for VMware Cloud Foundation (VCF) 5.2.x environment.

In [ ]:
if ! command -v "expect" &> /dev/null; then
    echo "Package 'expect' not found, installing..."
    sudo tdnf install -y "expect"
else
    echo "Package 'expect' is already installed. Skipping."
fi
# To download all the required bundles, specify value as: all.
# To download all install bundles, specify value as: install.
# To download all patch bundles, specify value as: patch.
/usr/bin/expect <<eof
spawn /root/vdt/bin/lcm-bundle-transfer-util --download  --outputDirectory /var/www/build/PROD/COMP/SDDC_MANAGER_VCF --depotDownloadActivationCodeFile /root/activation_code.txt --productVersion $VCF_VERSION_LEGACY
expect "continue: "
send "y\r"
expect "bundles (Y/N)?: "
send "y\r"
expect -timeout 200 "quit/q"
send "install\r"
expect -timeout 10000 "Successfully completed downloading all VSRN bundles to directory path"
eof
echo "Latest VCF $VCF_VERSION_LEGACY bundles and manifests downloaded.  Don't forget to update the index."
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nVCF 5.2.x bundles and manifests download are completed." | msmtp $Email_To
fi

[Return to Top of Page](#Home)

<a id="DL5_Index"></a>
### Update downloaded bundles index
Laslty, you need to update the index.v3 file after using the VMware Cloud Foundation (VCF) Offline Bundle Transfer Utility (OBTU) to ensure the local SDDC Manager inventory recognizes newly uploaded update bundles.

In [ ]:
# Update index.v3 based on actual downloaded bundles and manifests
cd /var/www/build/PROD/COMP/SDDC_MANAGER_VCF
rm index.v3
for i in $(ls bundles);
do
    BUNDLE=$(echo ${i%.tar})
    grep ${BUNDLE} tmp/index.v3 >> index.v3
done
#
echo "VCF $VCF_VERSION_LEGACY repository index updated."

[Return to Top of Page](#Home)

<a id="DL_Ext"></a>
## Downloading 9.x binaries to local depot

The following cells use the VCF Download Tool (VCFDT) to download the required VCF 9.x installation binaries, patches, and component updates to the local offline depot, supporting both convenience and fully air‑gapped environments.
Begin by updating the Broadcom external/public repository URL for your VCF 9.x downloads.

In [ ]:
# Legacy VDT 9.0.2 requirement
# sed -i -e "s|lcm.depot.adapter.host=.*|lcm.depot.adapter.host=$LCM_HOST|" /root/vdt/conf/application-prodv2.properties

In [9]:
sed -i -e "s|lcm.depot.adapter.host=.*|lcm.depot.adapter.host=$LCM_HOST|" /root/vdt/conf/application-prodv2.properties
sed -i -e "s|lcm.depot.adapter.enableBundleSignatureValidation=.*|lcm.depot.adapter.enableBundleSignatureValidation=false|" /root/vdt/conf/application-prodv2.properties
grep -qx "lcm.depot.adapter.enableSignatureValidation=false" /root/vdt/conf/application-prodv2.properties || echo "lcm.depot.adapter.enableSignatureValidation=false" >> /root/vdt/conf/application-prodv2.properties
grep -qx "lcm.esx.download.max.retries=10" /root/vdt/conf/application-prodv2.properties || echo "lcm.esx.download.max.retries=10" >> /root/vdt/conf/application-prodv2.properties
grep -qx "lcm.access_token.broadcom.authorization.server.url=https://eapi-gcpstg.broadcom.com/vcf/generateToken" /root/vdt/conf/application-prodv2.properties || echo "lcm.access_token.broadcom.authorization.server.url=https://eapi-gcpstg.broadcom.com/vcf/generateToken" >> /root/vdt/conf/application-prodv2.properties
echo "Configure VCF Download Tool to Broadcom staging repository."

Setting the latest Broadcom Repository URL for VCF Download Tool.


We can now begin downloading the required binaries. Depending on your workflow, you may choose to list or download only the minimum files needed for installation, the full set of files, or specific components. The following sections outline the most commonly used actions.<br>
<b>These blocks are not intended to be executed sequentially—select the one that aligns with your use case or adjust as needed.</b>

<div class="alert alert-block alert-info">
<b>Tip:</b> The download operations can take significant time to complete. Please be patient.
</div>

[Return to Top of Page](#Home)

<a id="DL_VCF_Installer"></a>
### List all binaries for a specific VCF version installation or upgrade
Execute the cell below to see the available binaries that are required for the initial installation or upgrade.<br>

In [ ]:
# Binaries required for fresh installation.
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries list --depot-download-activation-code-file=/root/activation_code.txt --vcf-version=$VCF_VERSION --automated-install --type=INSTALL --sku=VCF --ceip=DISABLE
done

In [ ]:
# Binaries required for upgrade to an existing install.
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries list --depot-download-activation-code-file=/root/activation_code.txt --vcf-version=$VCF_VERSION --automated-install --type=UPGRADE --sku=VCF --ceip=DISABLE
done

[Return to Top of Page](#Home)

<a id="DL_VCF_Installer"></a>
### Download all binaries needed by the VCF Installer for the specified VCF releases
Most commonly used to support the VCF 9.x installer, ideal for fresh installations.<br>
<div class="alert alert-block alert-info">
<b>VCF 9.0.x - </b>(NSX_T_MANAGER, VRSLCM, SDDC_MANAGER_VCF, VCF_OPS_CLOUD_PROXY, VROPS, VCENTER, VRA)<br>
<b>VCF 9.1.x - </b>(VSP, VCENTER, NSX_T_MANAGER, DEPOT_SERVICE, TELEMETRY_ACCEPTOR, VCF_OPS_CLOUD_PROXY, VCF_SALT, VCF_SERVICE_VCD_MIGRATION_BACKEND, SDDC_MANAGER_VCF, VROPS, VCF_SDDC_LCM, VRA, VIDB, VCF_SALT_RAAS, VCF_LICENSE_SERVER, VCF_FLEET_LCM)
</div>
Note: For token use case "--depot-download-token-file=/root/download_token.txt"

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    start_ts=$(TZ="$MyTimeZone" date '+%Y-%m-%d %H:%M:%S %Z')
    echo "[$start_ts] Starting download for VCF version: $VCF_VERSION - VCF Installer required components" | tee -a "$VDT_LOGFILE"
    if /root/vdt91/bin/vcf-download-tool binaries download \
        --depot-download-activation-code-file=/root/activation_code.txt \
        --depot-store=/var/www/build \
        --vcf-version=$VCF_VERSION \
        --automated-install --type=INSTALL \
        --sku=VCF \
        --ceip=DISABLE; then
        end_ts=$(TZ="$MyTimeZone" date '+%Y-%m-%d %H:%M:%S %Z')
        echo "[$end_ts] SUCCESS: VCF $VCF_VERSION - VCF Installer required components downloaded successfully" | tee -a "$VDT_LOGFILE"
    else
        end_ts=$(date '+%Y-%m-%d %H:%M:%S %Z')
        echo "[$end_ts] FAILURE: VCF $VCF_VERSION - VCF Installer required components download failed" | tee -a "$VDT_LOGFILE"
    fi
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nVCFDT download activities for the VCF Installer and all associated dependencies are completed." | msmtp $Email_To
fi

[Return to Top of Page](#Home)

<a id="DL_ESX"></a>
### Download the ESXi binaries required for Holodeck
This step must be executed alongside the previous instructions to enable Holodeck to deploy a nested environment.

In [ ]:
component="ESX_HOST"
for VCF_VERSION in "${VCF_versions[@]}"; do
    start_ts=$(TZ="$MyTimeZone" date '+%Y-%m-%d %H:%M:%S %Z')
    echo "[$start_ts] Starting download for VCF version: $VCF_VERSION - $component" | tee -a "$VDT_LOGFILE"

    if /root/vdt91/bin/vcf-download-tool binaries download \
        --component=$component \
        --depot-download-activation-code-file=/root/activation_code.txt \
        --depot-store=/var/www/build \
        --vcf-version=$VCF_VERSION \
        --ceip=DISABLE; then
        end_ts=$(TZ="$MyTimeZone" date '+%Y-%m-%d %H:%M:%S %Z')
        echo "[$end_ts] SUCCESS: VCF $VCF_VERSION - $component downloaded successfully" | tee -a "$VDT_LOGFILE"
    else
        end_ts=$(date '+%Y-%m-%d %H:%M:%S %Z')
        echo "[$end_ts] FAILURE: VCF $VCF_VERSION - $component download failed" | tee -a "$VDT_LOGFILE"
    fi
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nVCFDT download activities for ESX is completed." | msmtp $Email_To
fi

[Return to Top of Page](#Home)

<a id="DL_VCF_ALL"></a>
### Download ALL binaries for specified VCF releases
Most commonly used when refreshing or deploying a new environment to build a brand‑new repository (depot).<br>
<div class="alert alert-block alert-info">
<b>VCF 9.0.x - </b>(NSX_T_MANAGER, VRSLCM, SDDC_MANAGER_VCF, VCF_OPS_CLOUD_PROXY, VROPS, VCENTER, VRA, VIDM, VRLI, VRNI, VCFDT, HCX, VMRC, VMTOOLS)<br>
<b>VCF 9.1.x - </b>(VSP, VCENTER, NSX_T_MANAGER, DEPOT_SERVICE, TELEMETRY_ACCEPTOR, VCF_OPS_CLOUD_PROXY, VCF_SALT, VCF_SERVICE_VCD_MIGRATION_BACKEND, SDDC_MANAGER_VCF, VROPS, VCF_SDDC_LCM, VRA, VIDB, VRNI, VCFDT, HCX, VCF_SALT_RAAS, VCF_LICENSE_SERVER, VCF_FLEET_LCM, VSAN_FILE_SERVICES, VCFMS_METRICS_STORE, VCF_OBSERVABILITY_DATA_PLATFORM)
</div>
Note: For token use case, replace "--depot-download-activation-code-file=/root/activation_code.txt" with "--depot-download-token-file=/root/download_token.txt"

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    for VCF_COMPONENT in "${VCF_components[@]}"; do
        start_ts=$(TZ="$MyTimeZone" date '+%Y-%m-%d %H:%M:%S %Z')
        echo "[$start_ts] Starting download for VCF version: $VCF_VERSION - $VCF_COMPONENT" | tee -a "$VDT_LOGFILE"
    
        if /root/vdt91/bin/vcf-download-tool binaries download \
            --component=$VCF_COMPONENT \
            --depot-download-activation-code-file=/root/activation_code.txt \
            --depot-store=/var/www/build \
            --vcf-version=$VCF_VERSION \
            --ceip=DISABLE; then
            end_ts=$(TZ="$MyTimeZone" date '+%Y-%m-%d %H:%M:%S %Z')
            echo "[$end_ts] SUCCESS: VCF $VCF_VERSION - $VCF_COMPONENT downloaded successfully" | tee -a "$VDT_LOGFILE"
        else
            end_ts=$(date '+%Y-%m-%d %H:%M:%S %Z')
            echo "[$end_ts] FAILURE: VCF $VCF_VERSION - $VCF_COMPONENT download failed" | tee -a "$VDT_LOGFILE"
        fi
    done
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nVCFDT download activities for ALL VCF binaries are completed." | msmtp $Email_To
fi

[Return to Top of Page](#Home)

<a id="DL_ADD"></a>
### Download individual binaries/components for VCF 9.x and 9.1.x.
* VIDM
* VRLI
* VRA
* VRO
* VRNI
* VCFDT
* HCX
* VMRC
* VMTOOLS

Execute each cell below if you want to download VCF components individually.

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VIDB --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for Identity Broker (VIDB) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
/root/vdt/bin/vcf-download-tool binaries list --depot-download-activation-code-file=/root/activation_code.txt  --vcf-version=5.2.3 --type=INSTALL
/usr/bin/expect <<eof
spawn /root/vdt/bin/vcf-download-tool binaries list --depot-download-activation-code-file=/root/activation_code.txt  --vcf-version=5.2.3 --type=INSTALL
expect "Do you want to enable CEIP(Y/N)?"
send "y\r"
expect "bundles (Y/N)?: "
send "y\r"
expect -timeout 200 "quit/q"
send "install\r"
expect -timeout 10000 "Successfully completed downloading all VSRN bundles to directory path"
eof
echo "Latest VCF $VCF_VERSION_LEGACY bundles and manifests downloaded.  Don't forget to update the index."


expect {
    "One-time Prompt Text" {
        send "yes\r"
        expect "Next Prompt"
    }
    timeout {
        # The prompt didn't appear, continue normally
    }
}

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VRLI --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for Log Management (VRLI) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VRA --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF Automation (VRA) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VRO --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF Operations orchestrator (VRO) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VRNI --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF Operations for networks (VRNI) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VCFDT --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF download tool (VCFDT) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=HCX --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF Operations HCX (HCX) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VMRC --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VMware Remote Console (VMRC) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VMTOOLS --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VMware Tools (VMTOOLS ) via VCFDT are now completed." | msmtp $Email_To
fi

[Return to Top of Page](#Home)

<a id="DL_ADD2"></a>
### Download additional individual binaries/components specifically for VCF 9.1.
* TELEMETRY_ACCEPTOR
* VSAN_OSA_WITNESS
* VSAN_ESA_WITNESS
* VSAN_FILE_SERVICES
* VSP
* DEPOT_SERVICE	
* VCF_LICENSE_SERVER
* VCF_FLEET_LCM
* VCF_SDDC_LCM
* VCF_CONSUMPTION_CLI
* VCF_CONSUMPTION_CLI_PLUGINS
* VCFMS_METRICS_STORE
* VCF_SERVICE_VCD_MIGRATION_BACKEND
* VCF_SALT
* VCF_SALT_RAAS
* VCF_OBSERVABILITY_DATA_PLATFORM
* VCF Operations fleet management (9.0 only)


In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VRSLCM --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF Operations fleet management (9.0 only) (VRSLCM) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=TELEMETRY_ACCEPTOR --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF Tanzu Application Platform Telemetry (TELEMETRY_ACCEPTOR) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VSAN_OSA_WITNESS --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF vSAN OSA Witness component (VSAN_OSA_WITNESS) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VSAN_ESA_WITNESS --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF vSAN ESA Witness component (VSAN_ESA_WITNESS) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VSAN_FILE_SERVICES --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for vSAN File Services (VSAN_FILE_SERVICES) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VSP --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for vSphere Supervisor Platform (VSP) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=DEPOT_SERVICE --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for Unified Software Depot Service (DEPOT_SERVICE) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VCF_LICENSE_SERVER --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF License Server (VCF_LICENSE_SERVER) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VCF_FLEET_LCM --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF Fleet Lifecycle Manager (VCF_FLEET_LCM) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VCF_SDDC_LCM --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF SDDC Lifecycle Manager (VCF_SDDC_LCM) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VCF_CONSUMPTION_CLI --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF Supervisor Consumption CLI component (VCF_CONSUMPTION_CLI) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VCF_CONSUMPTION_CLI_PLUGINS --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF Supervisor Consumption CLI Plugins (VCF_CONSUMPTION_CLI_PLUGINS) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VCFMS_METRICS_STORE --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF Management Services Metrics Store (VCFMS_METRICS_STORE) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VCF_SERVICE_VCD_MIGRATION_BACKEND --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF Service VCD Migration Backend (VCF_SERVICE_VCD_MIGRATION_BACKEND) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VCF_OBSERVABILITY_DATA_PLATFORM --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF Operations Observability Data Platform (VCF_OBSERVABILITY_DATA_PLATFORM) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VCF_SALT --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF Salt Master aka Tanzu Salt (VCF_SALT) via VCFDT are now completed." | msmtp $Email_To
fi

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    /root/vdt91/bin/vcf-download-tool binaries download --component=VCF_SALT_RAAS --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --ceip=DISABLE
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for VCF SALT RaaS - Returner as a Service (VCF_SALT_RAAS) via VCFDT are now completed." | msmtp $Email_To
fi

[Return to Top of Page](#Home)

<a id="UG_SDDC"></a>
### Download upgrade binaries whose lifecycle is managed by SDDC Manager for a specific VCF version

In [ ]:
# /root/vdt/bin/vcf-download-tool binaries download --depot-download-token-file /root/download_token.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --lifecycle-managed-by=SDDC_MANAGER_VCF --type=UPGRADE --ceip=DISABLE

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    start_ts=$(TZ="$MyTimeZone" date '+%Y-%m-%d %H:%M:%S %Z')
    echo "[$start_ts] Starting download for VCF version: $VCF_VERSION - upgrade and patch binaries are managed through SDDC Manager." | tee -a "$VDT_LOGFILE"

    if /root/vdt91/bin/vcf-download-tool binaries download \
        --depot-download-activation-code-file=/root/activation_code.txt \
        --depot-store=/var/www/build \
        --vcf-version=$VCF_VERSION \
        --lifecycle-managed-by=SDDC_MANAGER_VCF \
        --type=UPGRADE \
        --sku=VCF \
        --ceip=DISABLE; then
        end_ts=$(TZ="$MyTimeZone" date '+%Y-%m-%d %H:%M:%S %Z')
        echo "[$end_ts] SUCCESS: VCF $VCF_VERSION - upgrades / patch binaries downloaded successfully" | tee -a "$VDT_LOGFILE"
    else
        end_ts=$(date '+%Y-%m-%d %H:%M:%S %Z')
        echo "[$end_ts] FAILURE: VCF $VCF_VERSION  - upgrades / patch binaries download failed" | tee -a "$VDT_LOGFILE"
    fi
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for upgrade and patch binaries managed through SDDC Manager are now completed." | msmtp $Email_To
fi

[Return to Top of Page](#Home)

<a id="UG_VRSLCM"></a>
### Download upgrade binaries whose lifecycle is managed by VRSLCM for a specific VCF version

In [ ]:
#/root/vdt/bin/vcf-download-tool binaries download --depot-download-token-file /root/download_token.txt --depot-store=/var/www/build --vcf-version=$VCF_VERSION --lifecycle-managed-by=VRSLCM --type=UPGRADE --ceip=DISABLE

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    start_ts=$(TZ="$MyTimeZone" date '+%Y-%m-%d %H:%M:%S %Z')
    echo "[$start_ts] Starting download for VCF version: $VCF_VERSION -  upgrade and patch binaries are managed through vRSLCM." | tee -a "$VDT_LOGFILE"

    if /root/vdt91/bin/vcf-download-tool binaries download \
        --depot-download-activation-code-file=/root/activation_code.txt \
        --depot-store=/var/www/build \
        --vcf-version=$VCF_VERSION \
        --lifecycle-managed-by=VRSLCM \
        --type=UPGRADE \
        --sku=VCF \
        --ceip=DISABLE; then
        end_ts=$(TZ="$MyTimeZone" date '+%Y-%m-%d %H:%M:%S %Z')
        echo "[$end_ts] SUCCESS: VCF $VCF_VERSION - upgrades / patch binaries downloaded successfully" | tee -a "$VDT_LOGFILE"
    else
        end_ts=$(date '+%Y-%m-%d %H:%M:%S %Z')
        echo "[$end_ts] FAILURE: VCF $VCF_VERSION  - upgrades / patch binaries download failed" | tee -a "$VDT_LOGFILE"
    fi
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for upgrade and patch binaries managed through vRSLCM are now completed." | msmtp $Email_To
fi

[Return to Top of Page](#Home)

<a id="UG_VRSLCM"></a>
### Download upgrade binaries whose lifecycle is managed by VCF Fleet Lifecycle Manager) for a specific VCF version

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    start_ts=$(TZ="$MyTimeZone" date '+%Y-%m-%d %H:%M:%S %Z')
    echo "[$start_ts] Starting download for VCF version: $VCF_VERSION -  upgrade and patch binaries are managed through VCF Fleet Lifecycle Manager." | tee -a "$VDT_LOGFILE"

    if /root/vdt91/bin/vcf-download-tool binaries download \
        --depot-download-activation-code-file=/root/activation_code.txt \
        --depot-store=/var/www/build \
        --vcf-version=$VCF_VERSION \
        --lifecycle-managed-by=VCF_FLEET_LCM \
        --type=UPGRADE \
        --sku=VCF \
        --ceip=DISABLE; then
        end_ts=$(TZ="$MyTimeZone" date '+%Y-%m-%d %H:%M:%S %Z')
        echo "[$end_ts] SUCCESS: VCF $VCF_VERSION - upgrades / patch binaries managed through VCF Fleet LCM downloaded successfully" | tee -a "$VDT_LOGFILE"
    else
        end_ts=$(date '+%Y-%m-%d %H:%M:%S %Z')
        echo "[$end_ts] FAILURE: VCF $VCF_VERSION  - upgrades / patch binaries managed through VCF Fleet LCM download failed" | tee -a "$VDT_LOGFILE"
    fi
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for upgrade and patch binaries managed through VCF Fleet Lifecycle Manager are now completed." | msmtp $Email_To
fi

[Return to Top of Page](#Home)

<a id="UG_VRSLCM"></a>
### Download upgrade binaries whose lifecycle is managed by SELF for a specific VCF version

In [ ]:
for VCF_VERSION in "${VCF_versions[@]}"; do
    start_ts=$(TZ="$MyTimeZone" date '+%Y-%m-%d %H:%M:%S %Z')
    echo "[$start_ts] Starting download for VCF version: $VCF_VERSION -  upgrade and patch binaries are managed through self." | tee -a "$VDT_LOGFILE"

    if /root/vdt91/bin/vcf-download-tool binaries download \
        --depot-download-activation-code-file=/root/activation_code.txt \
        --depot-store=/var/www/build \
        --vcf-version=$VCF_VERSION \
        --lifecycle-managed-by=SELF \
        --type=UPGRADE \
        --sku=VCF \
        --ceip=DISABLE; then
        end_ts=$(TZ="$MyTimeZone" date '+%Y-%m-%d %H:%M:%S %Z')
        echo "[$end_ts] SUCCESS: VCF $VCF_VERSION - upgrades / patch binaries downloaded successfully" | tee -a "$VDT_LOGFILE"
    else
        end_ts=$(date '+%Y-%m-%d %H:%M:%S %Z')
        echo "[$end_ts] FAILURE: VCF $VCF_VERSION  - upgrades / patch binaries download failed" | tee -a "$VDT_LOGFILE"
    fi
done
if [ -f "/etc/msmtprc" ]; then
    printf "Subject:VMware ODA Notification\n\nDownload operations for upgrade and patch binaries managed through self are now completed." | msmtp $Email_To
fi

[Return to Top of Page](#Home)

<a id="vLCM"></a>
## Update vSphere Lifecycle Download (vLCM) source(s)
vSphere Lifecycle Manager (vLCM) primarily sources ESXi and vCenter patches from the official Broadcom online depot via unique, token-based URLs

<a id="Update_vLCM"></a>
### Update Lifecycle Download Source(s) on vCenter
The following cell replaces the deprecated vmware.com repository URLs on vCenter with the updated, authenticated broadcom.com endpoints derived from your unique Download Tokens.

In [ ]:
SessionID=$(curl -k -s -X POST -u "$vCenterUser:$vCenterPassword" "https://$vCenterFQDN/api/session" -H 'Accept: application/json' | tr -d '"')
if [ -n "$SessionID" ]; then
    echo "Successfully obtained Session ID: $SessionID"
else
    echo "Authentication failed.  Failed to obtain Session ID."
fi
Source_IDS=$(curl -k -s -X GET "https://$vCenterFQDN/api/esx/settings/depots/online" -H "vmware-api-session-id: $SessionID" -H "Accept: application/json" \
    | jq -r 'to_entries | map(select(.value.location | contains("broadcom"))) | .[].key')
Source_URLS=$(curl -k -s -X GET "https://$vCenterFQDN/api/esx/settings/depots/online" -H "vmware-api-session-id: $SessionID" -H "Accept: application/json" \
    | jq -r 'to_entries | map(select(.value.location | contains("broadcom"))) | .[].value.location')
if [ -z "$Source_URLS" ]; then
    echo "No Lifecycle Download Source(s) found that can be delete."
else
    echo "The following Lifecycle Download Source(s) will be deleted from vCenter - $vCenterFQDN."
    printf "%s\n" $Source_URLS

    printf "%s\n" "${Source_IDS[@]}" | while read -r SourceID; do
    Delete_IDS=$(curl -s -k -w "%{http_code}" -o /dev/null -X DELETE -H "vmware-api-session-id: $SessionID" \
        "https://$vCenterFQDN/api/esx/settings/depots/online/$SourceID")
  
    if [ "$Delete_IDS" -eq 204 ]; then
        echo "Successfully deleted download source - $SourceID."
    else
        echo "Failed to delete source - $SourceID. HTTP Status: $Delete_IDS"
    fi
    done
fi

New_Source_URL1=$(curl -k  -w "%{http_code}" -o /dev/null -X POST "https://$vCenterFQDN/api/esx/settings/depots/online" -H "vmware-api-session-id: $SessionID" -H "Content-Type: application/json"\
  -d '{ "description": "Download vSphere ESXi and ESX patches", "enabled": true, "location": "https://'"$LCM_HOST"'/'"$DOWNLOAD_TOKEN"'/PROD/COMP/ESX_HOST/main/vmw-depot-index.xml", "ownerdata": "" }')
    if [ $New_Source_URL1 -eq 201 ]; then
        echo "Successfully added new download source for vSphere ESXi and ESX patches."
    elif [ $New_Source_URL1 -eq 400 ]; then
        echo "Adding new download source for vSphere ESXi and ESX patches skipped.  Prior entry already exists."
    else
        echo "Failed to add new download source for vSphere ESXi and ESX patches. HTTP Status: $New_Source_URL1"
    fi
New_Source_URL2=$(curl -k  -w "%{http_code}" -o /dev/null -X POST "https://$vCenterFQDN/api/esx/settings/depots/online" -H "vmware-api-session-id: $SessionID" -H "Content-Type: application/json"\
  -d '{ "description": "Partner provided Addons for ESXi", "enabled": true, "location": "https://'"$LCM_HOST"'/'"$DOWNLOAD_TOKEN"'/PROD/COMP/ESX_HOST/addon-main/vmw-depot-index.xml", "ownerdata": "" }')
    if [ $New_Source_URL2 -eq 201 ]; then
        echo "Successfully added new download source for Partner provided Addons for ESXi."
    elif [ $New_Source_URL2 -eq 400 ]; then
        echo "Adding new download source for Partner provided Addons for ESXi skipped.  Prior entry already exists."
    else
        echo "Failed to add new download source for Partner provided Addons for ESXi. HTTP Status: $New_Source_URL2"
    fi
 New_Source_URL3=$(curl -k  -w "%{http_code}" -o /dev/null -X POST "https://$vCenterFQDN/api/esx/settings/depots/online" -H "vmware-api-session-id: $SessionID" -H "Content-Type: application/json"\
  -d '{ "description": "VMware Certified Async Drivers for ESXi", "enabled": true, "location": "https://'"$LCM_HOST"'/'"$DOWNLOAD_TOKEN"'/PROD/COMP/ESX_HOST/iovp-main/vmw-depot-index.xml", "ownerdata": "" }')
    if [ $New_Source_URL3 -eq 201 ]; then
        echo "Successfully added new download source for VMware Certified Async Drivers for ESXi."
    elif [ $New_Source_URL3 -eq 400 ]; then
        echo "Adding new download source for VMware Certified Async Drivers for ESXi skipped.  Prior entry already exists."
    else
        echo "Failed to add new download source for VMware Certified Async Drivers for ESXi. HTTP Status: $New_Source_URL3"
    fi
 New_Source_URL4=$(curl -k  -w "%{http_code}" -o /dev/null -X POST "https://$vCenterFQDN/api/esx/settings/depots/online" -H "vmware-api-session-id: $SessionID" -H "Content-Type: application/json"\
  -d '{ "description": "VMware Async Releases for VM-tools on ESXi", "enabled": true, "location": "https://'"$LCM_HOST"'/'"$DOWNLOAD_TOKEN"'/PROD/COMP/ESX_HOST/vmtools-main/vmw-depot-index.xml", "ownerdata": "" }')
    if [ $New_Source_URL4 -eq 201 ]; then
        echo "Successfully added new download source for VMware Async Releases for VM-tools on ESXi."
    elif [ $New_Source_URL4 -eq 400 ]; then
        echo "Adding new download source for VMware Async Releases for VM-tools on ESXi skipped.  Prior entry already exists."
    else
        echo "Failed to add new download source for VMware Async Releases for VM-tools on ESXi. HTTP Status: $New_Source_URL4"
    fi  

[Return to Top of Page](#Home)

<a id="Sync_vLCM"></a>
### Initiate Sync Updates
Start synchronization between vCenter and assign online patch sources.

In [ ]:
SessionID=$(curl -k -s -X POST -u "$vCenterUser:$vCenterPassword" "https://$vCenterFQDN/api/session" -H 'Accept: application/json' | tr -d '"')
if [ -n "$SessionID" ]; then
    echo "Successfully obtained Session ID: $SessionID"
else
    echo "Authentication failed.  Failed to obtain Session ID."
fi
curl -k -X POST "https://$vCenterFQDN/api/esx/settings/depots?action=sync&vmw-task=true" -H "vmware-api-session-id: $SessionID" -H "Content-type: application/json"
echo "Online Patches sync Initiated"

[Return to Top of Page](#Home)

<a id="UMDS"></a>
## VMware UMDS (Update Manager Download Service)
In vCenter Server deployments without access to the Internet, instead of synchronizing to online depots, you can configure vSphere Lifecycle Manager to download updates from a UMDS-created shared repository.

<div class="alert alert-block alert-info">
<b>Note:</b> Due to licensing restriction, VMware no longer distributes UMDS as a standalone public download. The VMware UMDS package (VMware‑UMDS‑8.0.3.00800‑15560290.tar.gz) must be manually extracted from the vCenter 8.0.3 ISO and uploaded to the /root directory on the ODA appliance.
</div>

<a id="NGINX_UMDS"></a>
### Configure NGINX web server for UMDS
Update NGINX configuration to host downloaded patch binaries, metadata, and compatibility bundles for ESX hosts.

In [ ]:
sed -i '/http\s*{/r'<(cat <<EOF
    server {
        listen       80;
        listen       443 ssl;
        server_name  umds;
        ssl_certificate  /root/certs/localhost.crt;
        ssl_certificate_key  /root/certs/localhost.key;
        
        location / {
            root   $UMDS8_PatchStore_Export;
            index  index.html index.htm;
            autoindex on;
        }
    }
EOF
) /etc/nginx/nginx.conf
sudo systemctl restart nginx
echo "NGINX updated to support UMDS."

[Return to Top of Page](#Home)

<a id="UMDS_Certificate"></a>
### Setup UMDS Self-Signing Certificate
Generate a self‑signed certificate, create a backup, and update the NGINX configuration to enable HTTPS for vSphere Update Manager Download Service (UMDS).

In [ ]:
if [[ -e "/etc/nginx/ssl/umds.$UMDS_SSL_CommonName.crt" ]] then
    echo "Looks like a self-signing certificate - '$UMDS_SSL_CommonName.crt' already exists at /etc/nginx/ssl."
    echo "If you think this is an error or you want to generate a new (or expired) certificate, please manually delete the certificate inside folder '/etc/nginx/ssl/' and try again."
else
    openssl req -x509 -nodes -days 365 -newkey rsa:2048 \
    -keyout /etc/nginx/ssl/$UMDS_SSL_CommonName.key -out /etc/nginx/ssl/$UMDS_SSL_CommonName.crt \
    -subj "/C=$SSL_Country/ST=$SSL_State/L=$SSL_Locality/O=$SSL_OrgName/OU=$SSL_OrgUnit/CN=$UMDS_SSL_CommonName/emailAddress=$SSL_eMail" \
    -addext "subjectAltName=DNS:$UMDS_SSL_SANName,IP:$SSL_SANIP" \
    -addext "keyUsage=critical,digitalSignature,keyCertSign" \
    -addext "extendedKeyUsage=serverAuth" \
    -addext "basicConstraints=critical,CA:TRUE"
    sed -i -e "s|server_name  umds|server_name  $UMDS_SSL_CommonName|" /etc/nginx/nginx.conf
    sed -i -e "s|/root/certs/localhost.crt|/etc/nginx/ssl/$UMDS_SSL_CommonName.crt|" /etc/nginx/nginx.conf
    sed -i -e "s|/root/certs/localhost.key|/etc/nginx/ssl/$UMDS_SSL_CommonName.key|" /etc/nginx/nginx.conf
    echo "New self-signing certificates generated for offline UMDS web service (nginx)."
    timestamp=$(date -u +"%Y-%m-%d_%H-%M")
    tar -czPf /root/umdsCerts_${timestamp}.tar.gz /etc/nginx/ssl/
    echo "UMDS Certificates are backed up on '/root' as 'umdsCerts_${timestamp}.tar.gz'.  Please download and save it to a secure location in case a restore is required."
fi

[Return to Top of Page](#Home)

<a id="UMDS_Certificate_Restore"></a>
### Restore previously saved UMDS Self-Signing Certificate (For rebuild)
<div class="alert alert-block alert-danger">
Restore a previously saved self‑signed umds certificate when performing a system recovery. Upload the saved certificate tarball (filename.tar.gz) to /root, and update the certFilename variable accordingly. Proceed with caution!!
</div>

In [ ]:
certFilename="umdsCerts_2026-05-05_01-10.tar.gz" # Update restore certifcate filename (time-stamp) accordingly i.e. umdsCerts_2026-04-24_01-22.tar.gz
if [ -d "/etc/nginx/ssl" ]; then
    echo "Nginx SSL directory already exists."
else
    echo "Nginx SSL directory does not exist. Creating directory."
    mkdir -p /etc/nginx/ssl
    chmod 700 /etc/nginx/ssl
fi
if [ -f "/root/$certFilename" ]; then
    echo "Save UMDS certificate tarball found."
    tar -xzvpf /root/$certFilename -C /
    echo "Save UMDS certificates (nginx) are restored."
    sed -i -e "s|server_name umds|server_name  $UMDS_SSL_CommonName|" /etc/nginx/nginx.conf
    sed -i -e "s|/root/certs/localhost.crt|/etc/nginx/ssl/$UMDS_SSL_CommonName.crt|" /etc/nginx/nginx.conf
    sed -i -e "s|/root/certs/localhost.key|/etc/nginx/ssl/$UMDS_SSL_CommonName.key|" /etc/nginx/nginx.conf
    echo "nginx configuration updated."
    sudo systemctl restart nginx
    echo "nginx service restarted." 
    
else
    echo "No saved UMDS certificate tarball was located."
fi

[Return to Top of Page](#Home)

<a id="UploadCert_UMDS"></a>
## Upload UMDS Certificate onto vCenter as Trusted Root (Optional)
Adding the newly created self-signing certificate (umds) to vCenter as a Trusted Root to enable HTTPS communication with vCenter Lifecycle Manager without errors.

In [ ]:
CertFile="/etc/nginx/ssl/$UMDS_SSL_CommonName.crt"

SessionID=$(curl -k -s -X POST -u "$vCenterUser:$vCenterPassword" "https://$vCenterFQDN/api/session" -H 'Accept: application/json' | tr -d '"')
if [ -n "$SessionID" ]; then
    echo "Successfully obtained Session ID: $SessionID"
else
    echo "Authentication failed.  Failed to obtain Session ID."
fi
CertData=$(sed -E ':a;N;$!ba;s/\r{0,1}\n/\\n/g' "$CertFile")

Payload=$(cat <<EOF
{
  "cert_chain": {
    "cert_chain": [ "$CertData" ]
  }
}
EOF
)
Response=$(curl -k -s -X POST "https://$vCenterFQDN/api/vcenter/certificate-management/vcenter/trusted-root-chains" \
    -H "vmware-api-session-id: $SessionID" -H "Content-Type: application/json" -d "$Payload")
if [ "$Response" == "400" ] || [ "$Response" == "401" ]; then

    echo "Failed to add certificate. HTTP Status: $Response"
else
    echo "Root certificate successfully added. - $Response"
fi

[Return to Top of Page](#Home)

<a id="UMDS8_Config"></a>
### Configure VMware UMDS 8
Install and configure VMware Update Manager Download Service (UMDS) 8.  Because of distribution licensing restrictions, the UMDS tarball (VMware‑UMDS‑8.0.3.#####‑########.tar.gz) must be manually uploaded to /root from your vCenter installation media.

In [ ]:
mkdir -p /var/log/vmware/vmware-updatemgr/umds

if [ -f "/root/vmware-umds8-distrib/bin/vmware-umds" ]; then 
    echo "VMware UMDS 8 already installed."
    /root/vmware-umds8-distrib/bin/vmware-umds -v
else
    if [ -f "/root/$UMDS_Tarball" ]; then
        echo "VMware UMDS 8 is not installed.  Initiate installation......."
        tar -xzf /root/$UMDS_Tarball -C /root # Upload umds tarball from your vCenter installation media to /root
        mv /root/vmware-umds-distrib/ /root/vmware-umds8-distrib/
    else
        echo "File '$UMDS_Tarball' does not exist. Please upload the proper UMDS tarball to /root on the depot appliance."
        exit 1
    fi
fi
# "/root/vdt/vmware-umds/vmware-uninstall-umds.pl". to uninstall it
sed -i -e "s|hostupdate.vmware.com/software/VUM/PRODUCTION/main/|$LCM_HOST/$DOWNLOAD_TOKEN/PROD/COMP/ESX_HOST/main/|" /root/vmware-umds8-distrib/bin/downloadConfig.xml
sed -i -e "s|hostupdate.vmware.com/software/VUM/PRODUCTION/addon-main/|$LCM_HOST/$DOWNLOAD_TOKEN/PROD/COMP/ESX_HOST/addon-main/|" /root/vmware-umds8-distrib/bin/downloadConfig.xml
sed -i -e "s|hostupdate.vmware.com/software/VUM/PRODUCTION/iovp-main/|$LCM_HOST/$DOWNLOAD_TOKEN/PROD/COMP/ESX_HOST/iovp-main/|" /root/vmware-umds8-distrib/bin/downloadConfig.xml
sed -i -e "s|hostupdate.vmware.com/software/VUM/PRODUCTION/vmtools-main/|$LCM_HOST/$DOWNLOAD_TOKEN/PROD/COMP/ESX_HOST/vmtools-main/|" /root/vmware-umds8-distrib/bin/downloadConfig.xml
#
sed -i -e "s|$LCM_HOST/.*/PROD/COMP/ESX_HOST/main/|$LCM_HOST/$DOWNLOAD_TOKEN/PROD/COMP/ESX_HOST/main/|" /root/vmware-umds8-distrib/bin/downloadConfig.xml
sed -i -e "s|$LCM_HOST/.*/PROD/COMP/ESX_HOST/addon-main/|$LCM_HOST/$DOWNLOAD_TOKEN/PROD/COMP/ESX_HOST/addon-main/|" /root/vmware-umds8-distrib/bin/downloadConfig.xml
sed -i -e "s|$LCM_HOST/.*/PROD/COMP/ESX_HOST/iovp-main/|$LCM_HOST/$DOWNLOAD_TOKEN/PROD/COMP/ESX_HOST/iovp-main/|" /root/vmware-umds8-distrib/bin/downloadConfig.xml
sed -i -e "s|$LCM_HOST/.*/PROD/COMP/ESX_HOST/vmtools-main/|$LCM_HOST/$DOWNLOAD_TOKEN/PROD/COMP/ESX_HOST/vmtools-main/|" /root/vmware-umds8-distrib/bin/downloadConfig.xml
#
#/root/vmware-umds8-distrib/bin/vmware-umds -S --token $DOWNLOAD_TOKEN
/root/vmware-umds8-distrib/bin/vmware-umds -S -P $UMDS8_PatchStore
/root/vmware-umds8-distrib/bin/vmware-umds -S -o $UMDS8_PatchStore_Export
/root/vmware-umds8-distrib/bin/vmware-umds -S --enable-host
/root/vmware-umds8-distrib/bin/vmware-umds -S --add-url https://vmwaredepot.dell.com/index.xml --url-type HOST
# /root/vdt/bin/vcf-download-tool umds run -S --add-url http://vibsdepot.hpe.com/index.xml --url-type HOST  # HP link based on google is not reachable
# /root/vdt/bin/vcf-download-tool umds run -S --add-url http://vibsdepot.hpe.com/index-drv.xml --url-type HOST  # HP link based on google is not reachable
#/root/vdt/bin/vcf-download-tool umds run -S -e embeddedEsx-8.0-INTL
#/root/vdt/bin/vcf-download-tool umds run -S -e esxio-8.0-INTL
/root/vmware-umds8-distrib/bin/vmware-umds -S -d embeddedEsx-7.0-INTL
/root/vmware-umds8-distrib/bin/vmware-umds -S -d embeddedEsx-6.7.0-INTL  
echo "VMware UMDS 8 configuration update is completed."

<a id="UMDS8_Update"></a>
### Download VMware UMDS 8 updates
Download the latest updates for the current UMDS 8 configuration, then export them afterward.

In [ ]:
/root/vmware-umds8-distrib/bin/vmware-umds -D
/root/vmware-umds8-distrib/bin/vmware-umds -E

[Return to Top of Page](#Home)

<a id="Switch_Depot"></a>
### Switching Patch Download Source to UMDS
Switch online patch sources to Offline UMDS Repository

In [ ]:
SessionID=$(curl -k -s -X POST -u "$vCenterUser:$vCenterPassword" "https://$vCenterFQDN/api/session" -H 'Accept: application/json' | tr -d '"')
if [ -n "$SessionID" ]; then
    echo "Successfully obtained Session ID: $SessionID"
else
    echo "Authentication failed.  Failed to obtain Session ID."
fi

curl -k -X PUT "https://$vCenterFQDN/api/esx/settings/depots/umds" -H "vmware-api-session-id: $SessionID" -H "Content-Type: application/json"\
  -d '{"description": "", "enabled": true, "location": "http://'$UMDS_SSL_CommonName'"}'
  echo "VMware Lifecycle Manager download source set to UMDS."
#  -d '{"description": "", "enabled": true, "location": "http://'$UMDS_SSL_CommonName'/'${UMDS8_PatchStore_Export##*/}/'"}'

[Return to Top of Page](#Home)

<a id="UMDS_Config"></a>
### Configure VMware UMDS 9
Install and configure VMware Update Manager Download Service (UMDS) 9.

In [42]:
# echo "obtu.telemetry.config=DISABLE" > /root/vdt91/conf/telemetry/telemetry.flag
/root/vdt91/bin/vcf-download-tool esx configuration -G
#/root/vdt91/bin/vcf-download-tool esx configuration -D embeddedEsx-6.7-INTL
#/root/vdt91/bin/vcf-download-tool esx configuration -D embeddedEsx-7.0-INTL
#/root/vdt91/bin/vcf-download-tool esx configuration -D esxio-8.0-INTL
#/root/vdt91/bin/vcf-download-tool esx configuration -D esxio-9.0-INTL
#/root/vdt91/bin/vcf-download-tool esx configuration -D esxio-9.1-INTL
#/root/vdt91/bin/vcf-download-tool esx download --depot-download-activation-code-file=/root/activation_code.txt --depot-store=/var/www/build

*********Welcome to VCF Download Tool***********

Version: 9.1.0.0.25371089
-------------------------------------------------------------------------------------------------------
URL Type | Removable | URL                                                                             
-------------------------------------------------------------------------------------------------------
HOST     | NO        | https://dl-pstg.broadcom.com/PROD/COMP/ESX_HOST/addon-main/vmw-depot-index.xml  
HOST     | NO        | https://dl-pstg.broadcom.com/PROD/COMP/ESX_HOST/main/vmw-depot-index.xml        
HOST     | NO        | https://dl-pstg.broadcom.com/PROD/COMP/ESX_HOST/iovp-main/vmw-depot-index.xml   
HOST     | NO        | https://dl-pstg.broadcom.com/PROD/COMP/ESX_HOST/vmtools-main/vmw-depot-index.xml
-------------------------------------------------------------------------------------------------------
4 elements                                                                                  

In [ ]:
# DOWNLOAD_TOKEN=$(cat /root/download_token2.txt)
if [ -f "/root/vdt/vmware-umds/bin/vmware-umds" ]; then 
    echo "VMware UMDS already installed."
    /root/vdt/vmware-umds/bin/vmware-umds -v
else
    echo "VMware UMDS is not installed.  Initiate instllation......."
    yes | /root/vdt/bin/vcf-download-tool umds install
fi
# "/root/vdt/vmware-umds/vmware-uninstall-umds.pl". to uninstall it
sed -i -e "s|dl.broadcom.com|$LCM_HOST|" /root/vdt/vmware-umds/bin/downloadConfig.xml
/root/vdt/bin/vcf-download-tool umds run -S --add-entitlement-token $DOWNLOAD_TOKEN
/root/vdt/bin/vcf-download-tool umds run -S -P $UMDS_PatchStore
/root/vdt/bin/vcf-download-tool umds run -S -o $UMDS_PatchStore_Export
/root/vdt/bin/vcf-download-tool umds run -S --enable-host
/root/vdt/bin/vcf-download-tool umds run -S --add-url https://vmwaredepot.dell.com/index.xml --url-type HOST
# /root/vdt/bin/vcf-download-tool umds run -S --add-url http://vibsdepot.hpe.com/index.xml --url-type HOST  # HP link based on google is not reachable
# /root/vdt/bin/vcf-download-tool umds run -S --add-url http://vibsdepot.hpe.com/index-drv.xml --url-type HOST  # HP link based on google is not reachable
/root/vdt/bin/vcf-download-tool umds run -S -d embeddedEsx-9.0-INTL
# /root/vdt/bin/vcf-download-tool umds run -S -e embeddedEsx-8.0-INTL
/root/vdt/bin/vcf-download-tool umds run -S -d esxio-9.0-INTL
# /root/vdt/bin/vcf-download-tool umds run -S -e esxio-8.0-INTL
/root/vdt/bin/vcf-download-tool umds run -S -d embeddedEsx-7.0-INTL
/root/vdt/bin/vcf-download-tool umds run -S -d embeddedEsx-7.0.0
echo "VMware UMDS configuration update is completed."

In [18]:
# DOWNLOAD_TOKEN=$(cat /root/download_token2.txt)
if [ -f "/root/vdt91/vmware-umds/bin/vmware-umds" ]; then 
    echo "VMware UMDS already installed."
    /root/vdt/vmware-umds/bin/vmware-umds -v
else
    echo "VMware UMDS is not installed.  Initiate instllation......."
    yes | /root/vdt91/bin/vcf-download-tool umds install
fi

VMware UMDS is not installed.  Initiate instllation.......
*********Welcome to VCF Download Tool***********

Version: 9.1.0.0.25371089
umds
The UMDS tool has been deprecated.
Please run 'vcf-download-tool esx' commands for downloading, importing or
exporting ESX bundles.
For usage, please run 'vcf-download-tool esx -h'

Log file: /root/vdt91/log/vdt.log


<a id="UMDS_Update"></a>
### Download UMDS 9 updates
Download the latest updates for the current UMDS 9 configuration, then export them afterward.

In [ ]:
/root/vdt/bin/vcf-download-tool umds run -D
/root/vdt/bin/vcf-download-tool umds run -E

[Return to Top of Page](#Home)

<a id="UMDS_Schedule"></a>
### Schedule UMDS updates (Optional)
Schedule UMDS download and export as a system cron job.

<div class="alert alert-block alert-danger">
WITH ACTIVE ENTITLEMENT (NON EXPIRING TOKEN) ONLY

In [ ]:
# jupyter lab --ZMQChannelsWebsocketConnection.iopub_data_rate_limit=1.0e10
# jupyter lab --ZMQChannelsWebsocketConnection.rate_limit_window=10
mkdir -p /root/cron
rm -f /root/cron/umds-*.sh
echo "/root/vmware-umds8-distrib/bin/vmware-umds -D" > /root/cron/umds8-download.sh
echo "/root/vmware-umds8-distrib/bin/vmware-umds -E" > /root/cron/umds8-export.sh
#
echo "/root/vdt/bin/vcf-download-tool umds run -D" > /root/cron/umds-download.sh
echo "/root/vdt/bin/vcf-download-tool umds run -E" > /root/cron/umds-export.sh
chmod 744 /root/cron/*
#
cat << EOF | sudo tee /etc/cron.d/umds >/dev/null
SHELL=/bin/bash
PATH=/sbin:/bin:/usr/sbin:/usr/bin
# MAILTO=""
# Job 1 - UMDS download every morning at 3:00 am
0 4 * * * * root /root/cron/umds8-download.sh >> /var/log/umds8-download.log 2>&1 | printf "Subject:$Email_Subject\n\nUMDS8 synchronization is completed." | msmtp $Email_To
# Job 2 - UMDS export every morning at 4:30 am
30 4  * * * root /root/cron/umds8-export.sh >> /var/log/umds8-export.log 2>&1 | printf "Subject:$Email_Subject\n\nUMDS8 export is completed." | msmtp $Email_To
EOF
sudo chmod 644 /etc/cron.d/umds
echo "UMDS cron jobs added."

[Return to Top of Page](#Home)

<a id="Set_Perms"></a>
## Setting Permissions

This task will ensure the permissions for the depot are set correctly.
You should use this task after you have downloaded new files to the depot or have manually added or modified the depot contents.

In [ ]:
chown -R nginx /var/www
chgrp -R nginx /var/www
chmod -R 750 /var/www
chmod g+s /var/www
echo "Files/Folders permission update for nginx."

[Return to Top of Page](#Home)

<a id="Stage_5"></a>
## Stage VCF 5.2.x binaries onto VMware Holodeck 9.x HoloRouter runtime folder

<div class="alert alert-block alert-info">
<b>Note:</b> The require ESX 8.0.3x iso and Cloud Builder Appliance 5.x ova must be upload onto the depot (/var/www/build/Legacy) manually before continuing.
</div>

In [ ]:
# Upload VMware ESX 8.0.3x iso and Cloud Builder Appliance 5.x ova onto Holodeck Holorouter runtime folders
j=0
for i in "${VCF5_versions[@]}"; do
    echo "Parsing VCF version: $i"
    CloudBuilder_file_path=$(find /var/www/build/Legacy/ -type f -name 'VMware-Cloud-Builder-'$i*)
    if [ -z "$CloudBuilder_file_path" ]; then
        echo "No Cloud Builder copied for VCF version $i."
    else
        CloudBuilder_filename=$(basename "$CloudBuilder_file_path")
        if [ $i == "5.2.0" ]; then
            i="5.2"
        fi
        if ! (sshpass -p "$HoloRouter_Pass" ssh root@$HoloRouter_FQDN "test -e /holodeck-runtime/bin/$i/$CloudBuilder_filename";) then
            echo Copying $CloudBuilder_filename to HoloRouter
            sshpass -p "$HoloRouter_Pass" scp $CloudBuilder_file_path root@$HoloRouter_FQDN:/holodeck-runtime/bin/$i/
            echo $CloudBuilder_file_path copied to HoloRouter
        else
            echo "File already exists, skips copy."
        fi
    fi
    esx_file_path=$(find /var/www/build/Legacy/ -type f -name 'VMware-VMvisor-Installer-'${VCF5_ESX_versions[$j]}*)
    if [ -z "$esx_file_path" ]; then
        echo "No ESX iso copied for VCF version $i."
    else
        esx_filename=$(basename "$esx_file_path")
        if [ $i == "5.2.0" ]; then
            i="5.2"
        fi
        if ! (sshpass -p "$HoloRouter_Pass" ssh root@$HoloRouter_FQDN "test -e /holodeck-runtime/bin/$i/$esx_filename";) then
            echo Copying $esx_filename to HoloRouter
            sshpass -p "$HoloRouter_Pass" scp $esx_file_path root@$HoloRouter_FQDN:/holodeck-runtime/bin/$i/
            echo $esx_file_path copied to HoloRouter
        else
            echo "File already exists, skips copy."
        fi
    fi
    ((j += 1))
done
echo "Files transfer completed."

[Return to Top of Page](#Home)

<a id="Stage_9"></a>
## Stage VCF 9.x binaries onto VMware Holodeck 9.x HoloRouter runtime folder

In [ ]:
# Upload VMware ESX 9.x iso and VCF SDDC Manager 9.x OVA onto Holodeck Holorouter runtime folders
for i in "${VCF9_versions[@]}"; do
    echo "Parsing VCF version: $i"
    esx_file_path=$(find /var/www/build/PROD/COMP/ -type f -name 'VMware-VMvisor-Installer-'$i*)
    if [ -z "$esx_file_path" ]; then
    echo "No ESX iso copied for VCF version $i."
    else
        esx_filename=$(basename "$esx_file_path")
        if ! (sshpass -p "$HoloRouter_Pass" ssh root@$HoloRouter_FQDN "test -e /holodeck-runtime/bin/$i/$esx_filename";) then
            echo Copying $esx_filename to HoloRouter.
            sshpass -p "$HoloRouter_Pass" scp $esx_file_path root@$HoloRouter_FQDN:/holodeck-runtime/bin/$i/
            echo $esx_file_path copied to HoloRouter.
        else
            echo "File already exists, skips copy."
        fi
    fi
    sddcmgr_file_path=$(find /var/www/build/PROD/COMP/ -type f -name 'VCF-SDDC-Manager-Appliance-'$i*)
    if [ -z "$sddcmgr_file_path" ]; then
    echo "No SDDC Manager copied for VCF version $i."
    else
        sddcmgr_filename=$(basename "$sddcmgr_file_path")
        if ! (sshpass -p "$HoloRouter_Pass" ssh root@$HoloRouter_FQDN "test -e /holodeck-runtime/bin/$i/$sddcmgr_filename";) then
            echo Copying $sddcmgr_filename to HoloRouter.   
            sshpass -p "$HoloRouter_Pass" scp $sddcmgr_file_path root@$HoloRouter_FQDN:/holodeck-runtime/bin/$i/
            echo $sddcmgr_file_path copied to HoloRouter.
        else
            echo "File already exists, skips copy."
        fi
    fi
done
echo "Files upload completed."

[Return to Top of Page](#Home)

<a id="Holodeck_Note"></a>
## Holodeck 9.0.2 Deployment Note
Environment variable and note for Holodeck deployment.  These are for example purposes only.

<table>
    <tr style="border-left: 2px solid black; border-right: 2px solid black; border-top: 2px solid black; border-bottom: 2px solid black; background-color: #10598A; font-size: 18px; font-weight: bold; color: white;">
        <th>Appliance</th>
        <th>FQDN</th>
        <th>Username</th>
        <th>Password</th>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-top: 2px solid black; border-bottom: 2px solid black; background-color: #7BF1A8; font-weight: bold; color: black; font-size: 18px;" align="center"; colspan=4>Management Domain</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;">VCF Installer / VCF Cloud Builder</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; background-color: #FEF9C2;">
            <a href="https://vcfinstaller-a.site-a.vcf.lab/vcf-installer-ui/login" style="color: black; text-decoration: none;">https://vcfinstaller-a.site-a.vcf.lab/</a>
        </td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; background-color: #FEF9C2;">admin@local (for VCF Installer)</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; background-color: #FEF9C2;">VMware123!VMware123!</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;"></td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;"></td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">admin (for VCF Cloud Builder)</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;"></td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;">VCF Operations</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">
            <a href="https://ops-a.site-a.vcf.lab/ui/login.action?vcf=1" style="color: black; text-decoration: none;">https://ops-a.site-a.vcf.lab/</a>
        </td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">admin or admin@local</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">VMware123!VMware123!</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;">VCF Automations</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">
            <a href="https://auto-a.site-a.vcf.lab/" style="color: black; text-decoration: none;">https://auto-a.site-a.vcf.lab/</a>
        </td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">admin</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">VMware123!VMware123!</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;"></td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">
            Organization Name
        </td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">system, org-mgmt-a or org-wld-a</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;"></td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;">ESX Hosts</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">
            <a href="https://esx-01a.site-a.vcf.lab/ui/#/login" style="color: black; text-decoration: none;">https://esx-01a.site-a.vcf.lab/</a>
        </td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">root</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">VMware123!VMware123!</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;"></td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">
            <a href="https://esx-02a.site-a.vcf.lab/ui/#/login" style="color: black; text-decoration: none;">https://esx-02a.site-a.vcf.lab/</a>
        </td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">root</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">VMware123!VMware123!</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;"></td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">
            <a href="https://esx-03a.site-a.vcf.lab/ui/#/login" style="color: black; text-decoration: none;">https://esx-03a.site-a.vcf.lab/</a>
        </td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">root</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">VMware123!VMware123!</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;"></td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">
            <a href="https://esx-04a.site-a.vcf.lab/ui/#/login" style="color: black; text-decoration: none;">https://esx-04a.site-a.vcf.lab/</a>
        </td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">root</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">VMware123!VMware123!</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;">Management vCenter</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">
            <a href="https://vc-mgmt-a.site-a.vcf.lab/ui/" style="color: black; text-decoration: none;">https://vc-mgmt-a.site-a.vcf.lab/ui/</a>
        </td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">administrator@vsphere.local</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">VMware123!VMware123!</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;">SDDC Manager</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">
            <a href="https://sddcmanager-a.site-a.vcf.lab/" style="color: black; text-decoration: none;">https://sddcmanager-a.site-a.vcf.lab/</a>
        </td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">administrator@vsphere.local</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">VMware123!VMware123!</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;">Management NSX</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">
            <a href="https://nsx-mgmt-a.site-a.vcf.lab/" style="color: black; text-decoration: none;">https://nsx-mgmt-a.site-a.vcf.lab/</a>
        </td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">admin</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">VMware123!VMware123!</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;"></td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;"></td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;"></td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;"></td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-top: 2px solid black; border-bottom: 2px solid black; background-color: #53EAFD; font-weight: bold; color: black; font-size: 18px;" align="center"; colspan=4>Workload Domain</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;">Workload vCenter</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; background-color: #FEF9C2;">
            <a href="https://vc-wld01-a.site-a.vcf.lab/ui/" style="color: black; text-decoration: none;">https://vc-wld01-a.site-a.vcf.lab/ui/</a>
        </td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; background-color: #FEF9C2;">administrator@wld.sso</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; background-color: #FEF9C2;">VMware123!VMware123!</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;"></td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;"></td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">if isolated SSO enabled - administrator@vsphere.local</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;"></td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;">Management NSX</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">
            <a href="https://nsx-wld01-a.site-a.vcf.lab/" style="color: black; text-decoration: none;">https://nsx-wld01-a.site-a.vcf.lab/</a>
        </td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">admin</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">VMware123!VMware123!</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;">ESX Hosts</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">
            <a href="https://esx-05a.site-a.vcf.lab/ui/#/login" style="color: black; text-decoration: none;">https://esx-05a.site-a.vcf.lab/</a>
        </td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">root</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">VMware123!VMware123!</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;"></td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">
            <a href="https://esx-06a.site-a.vcf.lab/ui/#/login" style="color: black; text-decoration: none;">https://esx-06a.site-a.vcf.lab/</a>
        </td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">root</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">VMware123!VMware123!</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;"></td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">
            <a href="https://esx-07a.site-a.vcf.lab/ui/#/login" style="color: black; text-decoration: none;">https://esx-07a.site-a.vcf.lab/</a>
        </td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">root</td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px dotted black; background-color: #FEF9C2;">VMware123!VMware123!</td>
    </tr>
    <tr>
        <td style="border-left: 2px solid black; border-bottom: 2px solid black; background-color: #E2E8F0; font-weight: bold; color: black;"></td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px solid black; background-color: #FEF9C2;"></td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px solid black; background-color: #FEF9C2;"></td>
        <td style="border-left: 2px solid black; border-right: 2px solid black; border-bottom: 2px solid black; background-color: #FEF9C2;"></td>
    </tr>
</table>

#### PowerShell Basic
pwsh (or /usr/bin/pwsh) to open PowerShell within Holorouter via SSH.
- Get-Module -listavailable -Name Holodeck
- Get-Command -Module Holodeck
#### Define Global Configuration (/holodeck-runtime/config/)
<span style="color:green">New-HoloDeckConfig -Description `<Description>` -TargetHost `<Target vCenter or ESX IP/FQDN>` -Username `<username>` -Password `<password>`</span><br>
<div class="alert alert-block alert-success"><b>i.e.</b> New-HoloDeckConfig -Description "VCF 9.0.2 - Full Stack" -TargetHost vc01.tataoui.com -UserName administrator@vsphere.local -Password VMware123!</div>

- $config (to check current)
- Get-HoloDeckConfig (to list all available)
- Import-HoloDeckConfig -ConfigID 1nps (to load)  
- Get-HolodeckConfig -ConfigId 1nps | Import-HolodeckConfig (to load)
- rm /holodeck-runtime/config/1nps (to delete)

Note: Master VCF dpeloyment json is stored in /holodeck-runtime/templates/config.json
#### Define Instances - Management Only (/holodeck-runtime/state/)
<span style="color:green">New-HoloDeckInstance -InstanceID 'vcf02' -Version 9.0.2.0 -ManagementOnly [-CIDR 10.1.0.0/20] [-vSANMode ESA] [-NsxEdgeClusterMgmtDomain] [-DeployVcfAutomation] [-DeploySupervisorMgmtDomain] [-LogLevel <String>] [-ProvisionOnly] [-VLANRangeStart <Int32[]>] [-DNSDomain vcf.lab] [-Site a] [-DepotType Offline] [-DeveloperMode] [<CommonParameters>]</span><br>
<div class="alert alert-block alert-success"><b>i.e.</b> New-HoloDeckInstance -InstanceID 'vcf02' -Version 9.0.2.0 -ManagementOnly -CIDR "10.1.0.0/20" -vSANMode ESA -NsxEdgeClusterMgmtDomain -VLANRangeStart 10 -DNSDomain vcf.lab -Site a -DepotType Offline -DeveloperMode</div>
- New-HoloDeckInstance -InstanceID 'vcf02' -ProvisionOnly -Version 9.0.2.0 -ManagementOnly -DepotType Offline  

#### Define Instances - Full Stack Deployment (/holodeck-runtime/state/)
<span style="color:green">New-HoloDeckInstance -Version <String> -InstanceID <String> [-CIDR <String[]>] [-vSANMode <String>] [-WorkloadDomainType <String>] [-NsxEdgeClusterMgmtDomain] [-NsxEdgeClusterWkldDomain] [-DeployVcfAutomation] [-DeploySupervisorWldDomain] [-DeploySupervisorMgmtDomain] [-LogLevel <String>] [-ProvisionOnly] [-VLANRangeStart <Int32[]>] [-DNSDomain <String>] [-Site <String>] [-DepotType <String>] [-DeveloperMode] [<CommonParameters>]</span>
<div class="alert alert-block alert-success"><b>i.e.</b> New-HoloDeckInstance -InstanceID 'vcf02' -Version 9.0.2.0 -CIDR "10.1.0.0/20" -vSANMode ESA -WorkloadDomainType SharedSSO -NsxEdgeClusterMgmtDomain -NsxEdgeClusterWkldDomain -DeployVcfAutomation -VLANRangeStart 10 -DNSDomain vcf.lab -Site a -DepotType Offline -DeveloperMode</div>

#### Stop VCF Instances
<span style="color:green">Stop-HoloDeckInstance [-InstanceID] <string> [-Force]</span>
<div class="alert alert-block alert-warning"><b>i.e.</b> Stop-HoloDeckInstance -InstanceID piez</div>

#### Start VCF Instances (Broken in version 9.0.2.  Start VM manually)
<span style="color:green">Start-HoloDeckInstance [-InstanceID] <string> [-Force]</span>
<div class="alert alert-block alert-warning"><b>i.e.</b> Start-HoloDeckInstance -InstanceID piez</div>

#### Add new nested ESX Host
<span style="color:green">New-HoloDeckESXiNodes -Nodes <No. of Nodes> -CPU <No. of vCPU> -MemoryInGb <Memory in GB> -Site <'a' or 'b'> -vSANMode <'ESA' or 'OSA'></span>
<div class="alert alert-block alert-warning"><b>i.e.</b> echo "" > $config.state<br>
    New-HoloDeckESXiNodes -Nodes 1 -CPU 4 -MemoryInGb 96 -Site a -vSANMode ESA<br>
(When ask for VIDomain, answer with either 'management' or workload')</div>

#### Add new cluster (Always 3 nodes)
<span style="color:green">Update-HoloDeckInstance -Site a -AdditionalCluster -VIDomain Management</span><br>
<span style="color:green">Update-HoloDeckInstance -Site a -AdditionalCluster -VIDomain</span><br>
<span style="color:green">Update-HoloDeckInstance -Site a -AddVcfAutomationAllAppsOrg -VIDomain Management</span>
<div class="alert alert-block alert-warning">VCF Automation and Supervisor must already be deployed (via -DeployVcfAutomation, -DeploySupervisorMgmtDomain and -DeploySupervisorWldDomain during New-HoloDeckInstance) before running -AddVcfAutomationAllAppsOrg.</div>

#### Remove Instances and Rest Network
<span style="color:green">Remove-HoloDeckInstance [-ResetHoloRouter]</span>
<div class="alert alert-block alert-danger"><b>i.e.</b> Remove-HoloDeckInstance -ResetHoloRouter</div>

#### Developer Mode
<span style="color:green">`$`env:brcm_build_token = "build"<br>
`$`env:enable_proxy = "y" or "n"<br>
`$`env:proxy_protocol = "http" or "https"<br>
`$`env:proxy_ip = "`<proxy_ip_address>`"<br>
`$`env:proxy_port = "`<proxy_port>`"<br>
`$`env:enable_proxy_auth = "y" or "n"<br>
`$`env:proxy_username = "`<proxy_username>`"<br>
`$`env:proxy_password = "`<proxy_password>`"<br>
`$`env:offline_depot_ip = "`<offline_depot_ip_address>`"<br>
`$`env:offline_depot_port = "`<offline_depot_port>`"<br>
`$`env:offline_depot_username = "`<offline_depot_username>`"<br>
`$`env:offline_depot_password = "`<offline_depot_password>`"<br>
`$`env:offline_depot_protocol = "http" or "https"<br>
`$`env:datastore_name = "`<datastore_name>`"<br>
`$`env:trunk_port_group_name = "`<trunk_port_group_name>`"<br>
`$`env:cluster_name = "`<cluster_name>`"<br>
`$`env:dc_name = "`<datacenter_name>`"</span>

Copy and paste on Holodeck console: - Site A<br>
`$`env:offline_depot_ip = "depot.tataoui.com"<br>
`$`env:offline_depot_port = "80"<br>
`$`env:offline_depot_username = "admin"<br>
`$`env:offline_depot_password = "password"<br>
`$`env:offline_depot_protocol = "http"<br>
`$`env:datastore_name = "ESX01-NVMe-01"<br>
`$`env:trunk_port_group_name = "pg_Holodeck_Site_A"<br>
`$`env:cluster_name = "Holodeck_1"<br>
`$`env:dc_name = "HomeLab"

Copy and paste on Holodeck console: - Site B<br>
`$`env:offline_depot_ip = "depot.tataoui.com"<br>
`$`env:offline_depot_port = "80"<br>
`$`env:offline_depot_username = "admin"<br>
`$`env:offline_depot_password = "password"<br>
`$`env:offline_depot_protocol = "http"<br>
`$`env:datastore_name = "ESX02-NVMe-01"<br>
`$`env:trunk_port_group_name = "pg_Holodeck_Site_B"<br>
`$`env:cluster_name = "Holodeck_2"<br>
`$`env:dc_name = "HomeLab"

#### Display Subnet and DNS
<span style="color:green">Get-HoloDeckSubnet -Site a | ft -AutoSize</span><br>
<span style="color:green">Get-HoloDeckOverlaySubnet -Site "a" | ft -AutoSize</span>

#### Get DNS/IP info to site 'a' or 'b'
<span style="color:green">Import-HoloDeckConfig -ConfigID <config_ID><br>
Get-HoloDeckAppNetwork -Site a<br>
Get-HoloDeckAppNetwork -Site a -Hostname router<br>
Get-HoloDeckAppNetwork -Site a -IP 10.1.1.10<br>
Get-HoloDeckAppNetwork -Site a -FQDN esx-01a.site-a.vcf.lab<br>
</span>

#### DNS Management
<span style="color:green">Get-HoloDeckDNSConfig -IP 10.1.1.1<br>
Set-HoloDeckDNSConfig -DNSRecord '10.1.1.201 harbor.site-a.vcf.lab'<br>
Set-HoloDeckDNSConfig -SearchDNSRecord '10.1.1.201 harbor.site-a.vcf.lab' -ReplaceDNSRecord '10.1.1.210 harbor.site-a.vcf.lab' -Update<br>
Remove-HoloDeckDNSConfig -DNSRecord '10.1.1.210 harbor.site-a.vcf.lab'<br>
</span>

[Return to Top of Page](#Home)

### Holodeck 9 Webtop

In [ ]:
echo "Click the link below to open up Holodeck Webtop."
echo "http://$HoloRouter_FQDN:30000/"

### Execute Holodeck task within JupyterLab (test only - DO NOT USE)

In [ ]:
sshpass -p "$HoloRouter_Pass" ssh root@$HoloRouter_FQDN "pwsh -Command 'Import-HoloDeckConfig -ConfigID lao0;Stop-HoloDeckInstance -InstanceID vcf02'"

<div class="alert alert-block alert-danger">
<b>STOP:</b> Administration maintainance only beyond this point. Use with caution!
</div>

<a id="DeleteCert"></a>
## Delete Depot Certificate from SDDC Manager
Run the following cell to remove the added 'depot' self‑signed certificate from the VCF SDDC Manager.

In [ ]:
HTTP_Code_Token=$(curl -sk -w "%{http_code}" -o /tmp/token_response.json -X POST https://$sddcManagerFQDN/v1/tokens \
    -H 'Content-Type:application/json' \
    -d '{"username": "'$sddcManagerUser'","password": "'$sddcManagerPass'"}')
echo "Access Token HTTP status: $HTTP_Code_Token"
AccessToken=$(jq -r '.accessToken' /tmp/token_response.json)
if [[ $HTTP_Code_Token != "200" || -z $AccessToken || $AccessToken == "null" ]]; then
    echo "Failed to retrieve Access Token (HTTP $HTTP_Code_Token)."
    cat /tmp/token_response.json
    exit 1
else
    echo "Access Token acquired."
fi

alias="vcf_$HOSTNAME"
echo "Fetching trusted certificates ..."
Cert_List=$(curl -sk -H "Authorization: Bearer $AccessToken" \
    "https://$sddcManagerFQDN/v1/sddc-manager/trusted-certificates")
mapfile -t MATCHES < <(echo "$Cert_List" | jq -r --arg p "$alias" '.elements[]?.alias | select(startswith($p))')
count=${#MATCHES[@]}
if [[ $count -eq 0 ]]; then
    echo "No aliases found starting with \"$alias\" — nothing to delete."
    exit 0
fi
echo "Found $count matching certificate(s):"
for a in "${MATCHES[@]}"; do
    echo "  • $a"
done

echo "Deleting certificate..."
for alias in "${MATCHES[@]}"; do
HTTP_Code_Delete=$(curl -sk -w "%{http_code}" -o /tmp/delete_${alias}.json \
    -X DELETE "https://$sddcManagerFQDN/v1/sddc-manager/trusted-certificates/$alias" \
    -H "Authorization: Bearer $AccessToken" \
    -H "Accept: application/json")

if [[ $HTTP_Code_Delete == "200" || $HTTP_Code_Delete == "202" || $HTTP_Code_Delete == "204" ]]; then
    echo "Certificate deleted: $alias (HTTP $HTTP_Code_Delete)"
else
    echo "Failed:  $alias (HTTP $HTTP_Code_Delete)"
    echo "     Response:"
    cat /tmp/delete_${alias}.json
fi
done

[Return to Top of Page](#Home)

<a id="DeleteCert_VC"></a>
## Delete Depot Certificate from vCenter
Run the following cell to remove the added 'depot' self‑signed certificate from the vCenter Certificate Store (Trusted Root).

In [ ]:
SessionID=$(curl -k -s -X POST -u "$vCenterUser:$vCenterPassword" "https://$vCenterFQDN/api/session" -H 'Accept: application/json' | tr -d '"')
if [ -n "$SessionID" ]; then
    echo "Successfully obtained Session ID: $SessionID"
else
    echo "Authentication failed.  Failed to obtain Session ID."
fi
CertChains=$(curl -k -s -X GET -H "vmware-api-session-id: $SessionID" \
    "https://$vCenterFQDN/api/vcenter/certificate-management/vcenter/trusted-root-chains" | jq -r '.[].chain')

for Chain_ID in $CertChains; do
    # Get details for the specific chain
    Chain_Info=$(curl -k -s -X GET -H "vmware-api-session-id: $SessionID" \
    "https://$vCenterFQDN/api/vcenter/certificate-management/vcenter/trusted-root-chains/$Chain_ID")

    # Extract certificate PEM and parse for Common Name using openssl
    CERT_PEM=$(echo "$Chain_Info" | jq -r '.cert_chain.cert_chain[0]')
    CURRENT_CN=$(echo "$CERT_PEM" | openssl x509 -noout -subject -nameopt RFC2253 | sed -n 's/^subject=.*CN=\([^,]*\).*/\1/p')

    if [ "$CURRENT_CN" == "$HOSTNAME" ]; then
        echo "Found matching Chain ID: $Chain_ID"
        Response=$(curl -s -k -X DELETE "https://$vCenterFQDN/api/vcenter/certificate-management/vcenter/trusted-root-chains/$Chain_ID" \
  -H "vmware-api-session-id: $SessionID" -w "%{http_code}")
        exit 0
    fi
done
echo "No root certificate found with Common Name: $HOSTNAME"

[Return to Top of Page](#Home)

<a id="DeleteCert_UMDS"></a>
## Delete UMDS Certificate from vCenter
Run the following cell to remove the added 'umds' self‑signed certificate from the vCenter Certificate Store (Trusted Root).

In [ ]:
SessionID=$(curl -k -s -X POST -u "$vCenterUser:$vCenterPassword" "https://$vCenterFQDN/api/session" -H 'Accept: application/json' | tr -d '"')
if [ -n "$SessionID" ]; then
    echo "Successfully obtained Session ID: $SessionID"
else
    echo "Authentication failed.  Failed to obtain Session ID."
fi
CertChains=$(curl -k -s -X GET -H "vmware-api-session-id: $SessionID" \
    "https://$vCenterFQDN/api/vcenter/certificate-management/vcenter/trusted-root-chains" | jq -r '.[].chain')

for Chain_ID in $CertChains; do
    # Get details for the specific chain
    Chain_Info=$(curl -k -s -X GET -H "vmware-api-session-id: $SessionID" \
    "https://$vCenterFQDN/api/vcenter/certificate-management/vcenter/trusted-root-chains/$Chain_ID")

    # Extract certificate PEM and parse for Common Name using openssl
    CERT_PEM=$(echo "$Chain_Info" | jq -r '.cert_chain.cert_chain[0]')
    CURRENT_CN=$(echo "$CERT_PEM" | openssl x509 -noout -subject -nameopt RFC2253 | sed -n 's/^subject=.*CN=\([^,]*\).*/\1/p')

    if [ "$CURRENT_CN" == "$UMDS_SSL_CommonName" ]; then
        echo "Found matching Chain ID: $Chain_ID"
        Response=$(curl -s -k -X DELETE "https://$vCenterFQDN/api/vcenter/certificate-management/vcenter/trusted-root-chains/$Chain_ID" \
  -H "vmware-api-session-id: $SessionID" -w "%{http_code}")
        exit 0
    fi
done
echo "No root certificate found with Common Name: $UMDS_SSL_CommonName"

[Return to Top of Page](#Home)

<a id="Expand_FS"></a>
## Increasing the Depot Storage

You might need to expand the filesystem of the depot as you populate it with more binaries. 

First, you need to increase the size of the physical disk. To do this, simply edit the size of the disk for the VM and increase it to what will suit your needs. It will support up to 5.9TB in size.
This can be done while the VM is powered on.

Next, we need to make sure the OS is aware of the space increase:

In [ ]:
echo 1 > /sys/block/sda/device/rescan

Finally, we will execute this command in order to resize the partition

In [ ]:
printf 'yes\n100%%\n' | parted /dev/sda resizepart 2 ---pretend-input-tty

You can verify that the partition has been resized by using the following command:

In [ ]:
parted -s -a opt /dev/sda "print free"

Next, you will need to expand the filesystem in order to allow the system to use the newly added partition space...

In [ ]:
resize2fs /dev/sda2

[Return to Top of Page](#Home)

<a id="BasicAuth"></a>
## Change the Basic Auth Credentials

The default username and password for the Basic Auth used by the web server is admin/password. You can easily change this by using the htpasswd command and specifying new credentials.

In [ ]:
htpasswd -bc /etc/nginx/.htpasswd admin password

#### For earlier OVA (0.13) users

In [ ]:
PassHash =${openssl password -apr1 'password'}
echo "admin:$PassHash" | sudo tee -a /etc/nginx/.htpasswd

#### How to manually sideload bundles onto SDDC Manager
https://datareload.com/manually-uploading-vcf-bundles/

In [ ]:
curl -k http://127.0.0.1/lcm/bundle/upload -X POST -d ‘{“bundle”:”/nfs/bundles/bundle-6718.tar”,”manifest”:”/nfs/bundles/bundle-6718.manifest”, “signature”:”/nfs/bundles/bundle-6718.manifest.sig”}’ -H ‘Content-Type:application/json’

[Return to Top of Page](#Home)